# **Flow Hydrograph GUI**

---
To use this code, play the cell using the play button at the top.

Once the GUI is displayed, simply put in the USGS station number. 

You can customize the figure by setting a log scale to a part or all of your graph. You must click the optional log scale button and specify the breaking point (in cfs) that you want the y axis to stop being log, then change the ratio you want your log axis to normal axis (top to bottom).

To compare a site, click on the optional "compare a second station" button. You can name these sites differentely to show up in the legend, or leave them as they are.

To save the figure, click the "save svg" button and choose the location you want your figured to be saved.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Annual Peak Flow Viewer — ipywidgets GUI
#  Single OR two-station comparison (side-by-side bars, per-station color wheel)
#  Works in both VS Code (Jupyter extension) and Jupyter Notebook/Lab
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
from matplotlib.colors import hex2color, to_hex
import colorsys
from io import StringIO
import ipywidgets as widgets
from IPython.display import display, clear_output
import tkinter as tk
from tkinter import filedialog

_LOGO_B64 = "..."


# ── Global font: Verdana (WRA style) ──────────────────────────────────────────
rcParams["font.family"]     = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"]  = 13
rcParams["axes.labelsize"]  = 11
rcParams["xtick.labelsize"] = 8
rcParams["ytick.labelsize"] = 8

# ── Return period config ──────────────────────────────────────────────────────
ALL_RP_TARGETS = [1.5, 2, 5, 10, 25, 50, 100]
ALL_RP_LABELS  = ["1.5-yr", "2-yr", "5-yr", "10-yr", "25-yr", "50-yr", "100-yr"]
RP_MIN_N = {"100-yr": 100, "50-yr": 50, "25-yr": 25,
            "10-yr": 10, "5-yr": 5, "2-yr": 2, "1.5-yr": 1}

def _active_rps(n):
    active_t, active_l = [], []
    for t, l in zip(ALL_RP_TARGETS, ALL_RP_LABELS):
        if n >= RP_MIN_N[l]:
            active_t.append(t)
            active_l.append(l)
    return active_t, active_l

# ── Color interpolation from a base hex color ─────────────────────────────────
def _build_palette(base_hex, n_steps=8):
    r, g, b = hex2color(base_hex)
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    palette = []
    for i in range(n_steps):
        t = i / (n_steps - 1)
        s_i = s * (1.0 - t * 0.75)
        v_i = v + (1.0 - v) * t * 0.9
        r_i, g_i, b_i = colorsys.hsv_to_rgb(h, s_i, min(v_i, 1.0))
        palette.append(to_hex((r_i, g_i, b_i)))
    return palette

def _interp_from_palette(palette, t):
    t = np.clip(t, 0.0, 1.0)
    idx  = t * (len(palette) - 1)
    lo   = int(idx)
    hi   = min(lo + 1, len(palette) - 1)
    frac = idx - lo
    c_lo = np.array(hex2color(palette[lo]))
    c_hi = np.array(hex2color(palette[hi]))
    rgb  = c_lo * (1 - frac) + c_hi * frac
    return to_hex(rgb)

def _bar_color_for(val, sorted_flows, palette):
    if pd.isna(val):
        return "#cccccc"
    if val >= sorted_flows[-1]:
        return _interp_from_palette(palette, 0.0)
    if val < sorted_flows[0]:
        return _interp_from_palette(palette, 1.0)
    for i in range(len(sorted_flows) - 1):
        lo_flow, hi_flow = sorted_flows[i], sorted_flows[i + 1]
        if lo_flow <= val < hi_flow:
            t_hi  = i       / (len(sorted_flows) - 1)
            t_lo  = (i + 1) / (len(sorted_flows) - 1)
            frac  = (val - lo_flow) / (hi_flow - lo_flow)
            t     = t_lo + frac * (t_hi - t_lo)
            return _interp_from_palette(palette, 1.0 - t)
    return _interp_from_palette(palette, 1.0)

# ── Helpers ───────────────────────────────────────────────────────────────────
def hydro_year(dt):
    return dt.year + 1 if dt.month >= 10 else dt.year

def fetch_data(station_id):
    urls_to_try = [
        f"https://nwis.waterdata.usgs.gov/usa/nwis/peak?site_no={station_id}&agency_cd=USGS&format=rdb",
        f"https://waterdata.usgs.gov/nwis/peak?site_no={station_id}&agency_cd=USGS&format=rdb",
    ]
    resp = None
    for url in urls_to_try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        if "peak_dt" in r.text or "peak_va" in r.text:
            resp = r
            break
    if resp is None:
        raise ValueError(
            f"Could not retrieve RDB peak-flow data for station {station_id}. "
            f"Tried both national USGS endpoints."
        )

    data_lines = [l for l in resp.text.splitlines()
                  if l.strip() and not l.startswith("#")]

    if len(data_lines) < 3:
        raise ValueError(f"No data returned for station {station_id}. "
                         "Check the station ID and try again.")

    header_line = data_lines[0]
    n_cols = len(header_line.split("\t"))

    format_row_idx = 1
    if all(part.strip().replace("s","").replace("d","").replace("n","").isdigit()
           for part in data_lines[1].split("\t") if part.strip()):
        format_row_idx = 1
    record_lines = data_lines[format_row_idx + 1:]

    good_lines = [l for l in record_lines if len(l.split("\t")) == n_cols]

    if not good_lines:
        raise ValueError(
            f"Station {station_id}: header has {n_cols} columns but no data "
            f"rows matched. First few raw lines:\n" + "\n".join(data_lines[:6])
        )

    clean = "\n".join([header_line] + good_lines)
    df = pd.read_csv(StringIO(clean), sep="\t", low_memory=False)

    if "peak_va" not in df.columns or "peak_dt" not in df.columns:
        raise ValueError(
            f"Expected columns 'peak_dt' and 'peak_va' not found for station "
            f"{station_id}. Columns present: {list(df.columns)}"
        )

    keep_cols = ["peak_dt", "peak_va"]
    for optional in ["peak_tm", "peak_cd", "gage_ht", "gage_ht_cd"]:
        if optional in df.columns:
            keep_cols.append(optional)

    df = df[keep_cols].copy()
    df = df.dropna(subset=["peak_va"])
    df["peak_va"] = pd.to_numeric(df["peak_va"], errors="coerce")
    df["peak_dt"] = pd.to_datetime(df["peak_dt"], errors="coerce")
    df = df.dropna(subset=["peak_va", "peak_dt"])
    df["year"] = df["peak_dt"].apply(hydro_year)
    return df

def compute_return_periods(df):
    n  = len(df)
    df = df.sort_values("peak_va", ascending=False).reset_index(drop=True)
    df["rank"]              = df.index + 1
    df["exceedance_prob_%"] = (df["rank"] / (n + 1)) * 100
    df["return_period_yr"]  = (n + 1) / df["rank"]
    return df

def build_df_full(df):
    df_annual = df.groupby("year", as_index=False)["peak_va"].max()
    all_years = pd.RangeIndex(df_annual["year"].min(), df_annual["year"].max() + 1)
    return pd.DataFrame({"year": all_years}).merge(df_annual, on="year", how="left")

# ── Widgets — Station 1 ───────────────────────────────────────────────────────
s1_header     = widgets.HTML("<b>Station 1</b>")
s1_id         = widgets.Text(description="Station ID:",
                              placeholder="e.g. 11113000",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="240px"))
s1_color      = widgets.ColorPicker(value="#004ca3", description="Color:",
                                    style={"description_width": "50px"},
                                    layout=widgets.Layout(width="160px"))
s1_label      = widgets.Text(value="", description="Label (opt):",
                              placeholder="Station 1 name",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="300px"))

# ── Widgets — Station 2 (optional) ───────────────────────────────────────────
s2_enable     = widgets.Checkbox(value=False, description="Compare a second station",
                                 style={"description_width": "initial"})
s2_box        = widgets.VBox([])

s2_id         = widgets.Text(value="", description="Station ID:",
                              placeholder="e.g. 11109000",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="240px"))
s2_color      = widgets.ColorPicker(value="#b83232", description="Color:",
                                    style={"description_width": "50px"},
                                    layout=widgets.Layout(width="160px"))
s2_label      = widgets.Text(value="", description="Label (opt):",
                              placeholder="Station 2 name",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="300px"))

def _toggle_s2(change):
    s2_box.children = (
        [widgets.HTML("<b>Station 2</b>"),
         widgets.HBox([s2_id, s2_color, s2_label])]
        if change["new"] else []
    )
s2_enable.observe(_toggle_s2, names="value")

# ── Shared settings ───────────────────────────────────────────────────────────
title_input   = widgets.Text(value="", description="Title (opt):",
                              placeholder="e.g. Santa Ynez River at Baptism Creek",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="500px"))
log_toggle    = widgets.Checkbox(value=True, description="Log scale (bottom panel)",
                                 style={"description_width": "initial"})
break_slider  = widgets.IntSlider(value=15000, min=1000, max=100000, step=1000,
                                  description="Break (cfs):",
                                  style={"description_width": "90px"},
                                  layout=widgets.Layout(width="420px"),
                                  readout_format=",d")
ratio_slider  = widgets.FloatSlider(value=0.6, min=0.1, max=0.9, step=0.05,
                                    description="Top panel %:",
                                    style={"description_width": "100px"},
                                    layout=widgets.Layout(width="380px"),
                                    readout_format=".0%")
run_button    = widgets.Button(description="Run",      button_style="primary",
                               layout=widgets.Layout(width="90px"))
save_button   = widgets.Button(description="Save SVG", button_style="success",
                               layout=widgets.Layout(width="100px"))
status_label  = widgets.Label(value="")
out           = widgets.Output()

ui = widgets.VBox([
    widgets.HTML(
        "<div style='display:flex;align-items:center;gap:14px;"
        "background:white;padding:10px 20px;border-radius:6px;"
        "border:2px solid #003B5C;margin-bottom:6px'>"
        "<img src='data:image/png;base64," + _LOGO_B64 + "' "
        "style='height:44px;width:auto'>"
        "<span style='color:#003B5C;font-size:16px;font-family:Verdana;font-weight:bold'>"
        "USGS Annual Peak Flow Viewer</span></div>"
    ),
    widgets.HTML("<hr style='margin:4px 0'>"),
    s1_header,
    widgets.HBox([s1_id, s1_color, s1_label]),
    s2_enable,
    s2_box,
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([title_input, log_toggle]),
    widgets.HBox([break_slider, ratio_slider]),
    widgets.HBox([run_button, save_button, status_label]),
    out,
])

_current_fig = None

# ── Core plot (handles 1 or 2 stations) ──────────────────────────────────────
def make_plot(datasets, use_log, break_val, custom_title="", axis_ratio=0.6):
    global _current_fig

    BREAK     = break_val
    all_min = min(d["df_full"]["year"].min() for d in datasets)
    all_max = max(d["df_full"]["year"].max() for d in datasets)
    all_years    = np.arange(all_min, all_max + 1)
    x_all        = np.arange(len(all_years))
    year_to_x    = {y: i for i, y in enumerate(all_years)}

    n_stations   = len(datasets)
    total_width  = 0.75
    bar_w        = total_width / n_stations
    offsets      = [(i - (n_stations - 1) / 2) * bar_w for i in range(n_stations)]

    global_max = max(d["df_full"]["peak_va"].max() for d in datasets)
    y_top_max  = global_max * 1.1

    fig = plt.figure(figsize=(18, 9))
    fig.patch.set_facecolor("white")
    top_r  = axis_ratio
    bot_r  = 1.0 - axis_ratio
    gs     = fig.add_gridspec(2, 1, height_ratios=[top_r, bot_r], hspace=0)
    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

    for d, offset in zip(datasets, offsets):
        df_full   = d["df_full"]
        palette   = _build_palette(d["base_color"])
        n         = d["n"]
        rp_targets, rp_labels = _active_rps(n)
        rp_flows  = {}
        for rp, lbl in zip(rp_targets, rp_labels):
            rp_flows[lbl] = np.interp(
                rp, d["df"]["return_period_yr"].iloc[::-1],
                d["df"]["peak_va"].iloc[::-1]
            )
        sorted_flows = [rp_flows[l] for l in rp_labels]

        for _, row in df_full.iterrows():
            if pd.isna(row["peak_va"]):
                continue
            xi    = year_to_x[row["year"]]
            color = _bar_color_for(row["peak_va"], sorted_flows, palette)
            for ax in (ax_top, ax_bot):
                ax.bar(xi + offset, row["peak_va"], width=bar_w * 0.92,
                       color=color, edgecolor="white", linewidth=0.2)

    # Draw asterisks for missing years pinned to the visual bottom of ax_bot.
    # Blended transform: x in data coords, y in axes fraction (0=bottom, 1=top).
    import matplotlib.transforms as mtransforms
    blended = mtransforms.blended_transform_factory(
        ax_bot.transData, ax_bot.transAxes
    )
    for d, offset in zip(datasets, offsets):
        for _, row in d["df_full"].iterrows():
            if pd.isna(row["peak_va"]):
                xi = year_to_x[row["year"]]
                ax_bot.text(xi + offset, 0.02, "*", ha="center", va="bottom",
                            fontsize=8, color=d["base_color"], fontweight="bold",
                            fontfamily="Verdana", transform=blended)

    all_bot_items = []
    all_top_items = []
    for d_idx, d in enumerate(datasets):
        d_n       = d["n"]
        d_rp_t, d_rp_l = _active_rps(d_n)
        d_rp_flows = {}
        for rp, lbl in zip(d_rp_t, d_rp_l):
            d_rp_flows[lbl] = np.interp(
                rp, d["df"]["return_period_yr"].iloc[::-1],
                d["df"]["peak_va"].iloc[::-1]
            )
        palette    = _build_palette(d["base_color"])
        base_color = d["base_color"]

        for lbl in d_rp_l:
            flow = d_rp_flows[lbl]
            ax   = ax_top if flow >= BREAK else ax_bot
            ax.axhline(y=flow, color=base_color, linewidth=1.2, linestyle=":", alpha=0.85)

        bot_scale  = "log" if use_log else "linear"
        bot_ylim   = (1 if use_log else 0, BREAK)
        bot_items  = [(d_rp_flows[l], l, base_color) for l in d_rp_l if d_rp_flows[l] <  BREAK]
        top_items  = [(d_rp_flows[l], l, base_color) for l in d_rp_l if d_rp_flows[l] >= BREAK]
        all_bot_items.extend(bot_items)
        all_top_items.extend(top_items)

    def _nudge_positions(items, ylim, yscale, min_gap_frac=0.03):
        """
        Given items = [(flow, label, color), ...], return adjusted y positions
        so that no two labels are closer than min_gap_frac * axis_span apart.
        Works in log space when yscale=='log'. Sorts by flow, nudges upward
        from bottom to separate collisions, then nudges downward from top to
        keep everything within the axis limits.
        """
        if not items:
            return []
        lo, hi = ylim
        if yscale == "log":
            import math
            lo_t = math.log10(max(lo, 1e-9))
            hi_t = math.log10(hi)
            span = hi_t - lo_t
            to_t   = lambda v: math.log10(max(v, 1e-9))
            from_t = lambda t: 10 ** t
        else:
            span = hi - lo
            to_t   = lambda v: v
            from_t = lambda t: t

        gap = min_gap_frac * span
        # Sort by flow ascending
        sorted_items = sorted(items, key=lambda x: x[0])
        positions = [to_t(flow) for flow, _, _ in sorted_items]

        # Forward pass: push up if too close to previous
        for i in range(1, len(positions)):
            if positions[i] - positions[i-1] < gap:
                positions[i] = positions[i-1] + gap

        # Backward pass: pull down if we've gone above the axis top
        hi_t_val = to_t(hi) - gap * 0.2
        for i in range(len(positions) - 1, -1, -1):
            if positions[i] > hi_t_val:
                positions[i] = hi_t_val
                hi_t_val -= gap
            else:
                break

        return [(from_t(pos), lbl, color)
                for pos, (_, lbl, color) in zip(positions, sorted_items)]

    def _draw_rp_annotations(ax, items, ylim, yscale):
        """
        Draw return-period labels as text annotations on the right side of ax,
        with a short leader line from the actual flow value to the nudged label.
        No twinx axis needed — everything is drawn directly onto ax.
        """
        if not items:
            return
        nudged = _nudge_positions(items, ylim, yscale)
        ax.tick_params(axis="y", which="both", right=False)
        for (y_text, lbl, color) in nudged:
            ax.annotate(
                lbl,
                xy=(1.01, y_text), xycoords=("axes fraction", "data"),
                fontsize=8, fontweight="bold", fontfamily="Verdana",
                color=color, va="center", ha="left",
                annotation_clip=False,
            )

    _draw_rp_annotations(ax_bot, all_bot_items,
                         (1 if use_log else 0, BREAK),
                         "log" if use_log else "linear")
    _draw_rp_annotations(ax_top, all_top_items,
                         (BREAK, y_top_max), "linear")

    if use_log:
        ax_bot.set_yscale("log")
        ax_bot.set_ylim(1, BREAK)
    else:
        ax_bot.set_ylim(0, BREAK)
    ax_top.set_ylim(BREAK, y_top_max)

    ax_top.spines["bottom"].set_visible(False)
    ax_bot.spines["top"].set_visible(False)
    ax_top.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

    ax_bot.set_xticks(x_all)
    ax_bot.set_xticklabels([str(y) for y in all_years],
                           fontsize=7.5, rotation=90, ha="center",
                           fontfamily="Verdana")
    ax_bot.set_xlabel("Hydrological Year", fontsize=11, fontfamily="Verdana")
    ax_bot.set_xlim(-0.8, len(all_years) - 0.2)

    title_str = (custom_title.strip() if custom_title.strip()
                 else "Annual Peak Flow — " +
                      " vs ".join(d["label"] or f"Station {d['station_id']}"
                                  for d in datasets))
    ax_top.set_title(title_str, fontsize=13, fontweight="bold", fontfamily="Verdana")
    fig.supylabel("Peak Discharge (cfs)", fontsize=11, fontfamily="Verdana", x=0.0)

    ax_top.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y):,}"))
    ax_bot.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y):,}"))
    for ax in (ax_top, ax_bot):
        ax.grid(axis="y", linestyle=":", alpha=0.4, which="both")
        ax.set_axisbelow(True)
        for lbl in ax.get_xticklabels() + ax.get_yticklabels():
            lbl.set_fontfamily("Verdana")

    legend_objects = []
    for d_idx, d in enumerate(datasets):
        palette    = _build_palette(d["base_color"])
        n          = d["n"]
        _, rp_lbls = _active_rps(n)
        max_lbl    = rp_lbls[-1]
        site_name  = d["label"] or f"Station {d['station_id']}"

        handles = []
        handles.append(mpatches.Patch(color=_interp_from_palette(palette, 0.0),
                                      label=f">{max_lbl}"))
        for i in range(len(rp_lbls) - 1, 0, -1):
            mid_t = 1.0 - ((i - 0.5) / (len(rp_lbls) - 1))
            handles.append(mpatches.Patch(color=_interp_from_palette(palette, mid_t),
                                          label=f"{rp_lbls[i-1]} – {rp_lbls[i]}"))
        handles.append(mpatches.Patch(color=_interp_from_palette(palette, 1.0),
                                      label=f"<{rp_lbls[0]}"))

        x_anchor = 0.01 + d_idx * 0.18
        leg = ax_top.legend(
            handles=handles,
            title=f"{site_name}\n(max: {max_lbl}, n={n})",
            fontsize=9.5, title_fontsize=9.5,
            loc="upper left",
            bbox_to_anchor=(x_anchor, 1.0),
            framealpha=0.9, handlelength=1.2,
        )
        for text in leg.get_texts():
            text.set_fontfamily("Verdana")
        leg.get_title().set_fontfamily("Verdana")
        ax_top.add_artist(leg)
        legend_objects.append(leg)

    ax_top.get_legend().remove() if ax_top.get_legend() not in legend_objects else None

    plt.tight_layout()
    _current_fig = fig
    plt.show()


# ── Summary table ─────────────────────────────────────────────────────────────
def print_summary(df, station_id, label=""):
    n = len(df)
    rp_targets, rp_labels_active = _active_rps(n)
    rp_names = [l.replace("-yr", "-Year") for l in rp_labels_active]
    name = label or f"Station {station_id}"
    print("=" * 58)
    print(f"  {name}  (n = {n} yrs)")
    print("=" * 58)
    print(f"  {'Event':<22} {'RP (yr)':>8} {'Flow (cfs)':>12}")
    print("-" * 58)
    for rp, lbl in zip(rp_targets, rp_names):
        flow = np.interp(rp, df["return_period_yr"].iloc[::-1],
                         df["peak_va"].iloc[::-1])
        print(f"  {lbl:<22} {rp:>8.1f}   {flow:>12,.0f}")
    print("=" * 58)

    base_cols   = ["rank", "year", "peak_dt", "peak_va",
                   "exceedance_prob_%", "return_period_yr"]
    base_names  = ["Rank", "Hydro Year", "Date", "Flow (cfs)",
                   "Exceedance Prob (%)", "Return Period (yr)"]
    if "gage_ht" in df.columns:
        base_cols.insert(4, "gage_ht")
        base_names.insert(4, "Gage Ht (ft)")

    display_df = df[base_cols].copy()
    display_df.columns = base_names
    display_df["Exceedance Prob (%)"] = display_df["Exceedance Prob (%)"].round(2)
    display_df["Return Period (yr)"]  = display_df["Return Period (yr)"].round(2)
    display(display_df)


# ── Button callbacks ───────────────────────────────────────────────────────────
def on_run(b):
    with out:
        clear_output(wait=True)
        use_log      = log_toggle.value
        break_val    = break_slider.value
        custom_title = title_input.value
        status_label.value = "Fetching data..."

        try:
            datasets = []

            df1 = fetch_data(s1_id.value.strip())
            df1 = compute_return_periods(df1)
            datasets.append({
                "df":         df1,
                "df_full":    build_df_full(df1),
                "station_id": s1_id.value.strip(),
                "base_color": s1_color.value,
                "label":      s1_label.value.strip(),
                "n":          len(df1),
            })

            if s2_enable.value and s2_id.value.strip():
                df2 = fetch_data(s2_id.value.strip())
                df2 = compute_return_periods(df2)
                datasets.append({
                    "df":         df2,
                    "df_full":    build_df_full(df2),
                    "station_id": s2_id.value.strip(),
                    "base_color": s2_color.value,
                    "label":      s2_label.value.strip(),
                    "n":          len(df2),
                })

            ns = " | ".join(f"{d['label'] or d['station_id']}: {d['n']} records"
                            for d in datasets)
            status_label.value = f"Done — {ns}"

            make_plot(datasets, use_log, break_val, custom_title,
                      axis_ratio=ratio_slider.value)

            for d in datasets:
                print_summary(d["df"], d["station_id"], d["label"])

        except Exception as e:
            status_label.value = f"Error: {e}"
            raise


def on_save(b):
    global _current_fig
    if _current_fig is None:
        status_label.value = "Run the plot first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save figure as SVG",
            defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            _current_fig.savefig(path, format="svg", bbox_inches="tight")
            status_label.value = f"Saved: {path}"
        else:
            status_label.value = "Save cancelled."
    except Exception as e:
        status_label.value = f"Save error: {e}"


run_button.on_click(on_run)
save_button.on_click(on_save)

# ── Launch ─────────────────────────────────────────────────────────────────────
display(ui)

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Annual Peak Flow & Average Flow Chart Generator
#  WRA — Riverscapes & Shorelines Team
#
#  Fetches USGS NWIS data for one or two stations and produces a dual-axis
#  grouped bar chart matching the Balance Hydrologics "Figure 3" style:
#    • Left y-axis  — Annual Peak Flow (cfs), dark navy bars
#    • Right y-axis — Annual Average Flow (cfs), light cyan bars
#    • Side-by-side bars per water year (grouped)
#    • Gold-bordered stats box (top-left) and source box (top-right)
#    • Optional second station for side-by-side comparison
#
#  Run in JupyterHub:
#    %run annual_flow_chart_gui.py
#  or open as a notebook cell with:
#    exec(open("annual_flow_chart_gui.py").read())
#
#  DEPENDENCIES:  pandas, numpy, matplotlib, requests, ipywidgets
#
#  USGS ENDPOINTS USED:
#    Peak flows:   https://nwis.waterdata.usgs.gov/usa/nwis/peak?site_no=...&format=rdb
#    Annual avg:   https://waterservices.usgs.gov/nwis/stat/?sites=...&statReportType=annual...
#    Site name:    https://waterservices.usgs.gov/nwis/site/?sites=...&format=rdb
# ═══════════════════════════════════════════════════════════════════════════════

import datetime as _dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import rcParams
import requests
from io import StringIO
import ipywidgets as widgets
from IPython.display import display, clear_output
import tkinter as tk
from tkinter import filedialog

_LOGO_B64 = "..."

# ── WRA brand palette ─────────────────────────────────────────────────────────
WRA_NAVY      = "#003B5C"
WRA_PEAK_BLUE = "#0071db"     # annual peak flow bars  (dark navy)
WRA_AVG_CYAN  = "#5BC8F5"     # annual average flow bars (light cyan)
WRA_GOLD      = "#C9A84C"     # annotation box border
WRA_GOLD_FILL = "#FFF3CD"     # annotation box fill
WRA_WHITE     = "#FFFFFF"
WRA_GRID      = "#D0D0D0"

# ── Verdana global defaults ───────────────────────────────────────────────────
rcParams["font.family"]     = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"]  = 13
rcParams["axes.labelsize"]  = 11
rcParams["xtick.labelsize"] = 9
rcParams["ytick.labelsize"] = 9


# ══════════════════════════════════════════════════════════════════════════════
#  USGS NWIS DATA FETCH
# ══════════════════════════════════════════════════════════════════════════════

def _water_year(dt):
    """Return USGS water year for a datetime (Oct 1 → Sep 30)."""
    return dt.year + 1 if dt.month >= 10 else dt.year


def fetch_peak_flows(site_no: str, start_wy: int, end_wy: int) -> pd.DataFrame:
    """
    Pull annual instantaneous peak flows from USGS NWIS.
    Returns DataFrame with columns [water_year, peak_flow_cfs].
    One row per water year — maximum peak if multiple peaks recorded.

    Endpoint:  nwis.waterdata.usgs.gov/usa/nwis/peak  (RDB format)
    Fallback:  waterdata.usgs.gov/nwis/peak           (same data, different host)

    Date filtering is applied post-download because the peak-flow service
    ignores begin_date/end_date parameters on some NWIS deployments.
    """
    _PEAK_URLS = [
        (f"https://nwis.waterdata.usgs.gov/usa/nwis/peak"
         f"?site_no={site_no}&agency_cd=USGS&format=rdb"),
        (f"https://waterdata.usgs.gov/nwis/peak"
         f"?site_no={site_no}&agency_cd=USGS&format=rdb"),
    ]

    resp     = None
    last_err = None
    for url in _PEAK_URLS:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            if "peak_dt" in r.text or "peak_va" in r.text:
                resp = r
                break
        except Exception as e:
            last_err = e

    if resp is None:
        raise RuntimeError(
            f"Could not retrieve peak flows for station {site_no}. "
            f"Last error: {last_err}"
        )

    # ── parse RDB ─────────────────────────────────────────────────────────────
    lines = [l for l in resp.text.splitlines()
             if not l.startswith("#") and l.strip()]
    if len(lines) < 3:
        return pd.DataFrame(columns=["water_year", "peak_flow_cfs"])

    header = lines[0]
    n_cols = len(header.split("\t"))
    # lines[1] is the format row (e.g. "5s\t10d\t15s…") — skip it
    data_lines = [l for l in lines[2:]
                  if not l.startswith("5s")
                  and len(l.split("\t")) == n_cols]
    if not data_lines:
        return pd.DataFrame(columns=["water_year", "peak_flow_cfs"])

    df = pd.read_csv(
        StringIO("\n".join([header] + data_lines)),
        sep="\t", low_memory=False,
    )

    if "peak_va" not in df.columns or "peak_dt" not in df.columns:
        raise ValueError(
            f"Expected columns 'peak_dt'/'peak_va' for station {site_no}. "
            f"Got: {list(df.columns)}"
        )

    df = df[["peak_dt", "peak_va"]].copy()
    df.columns          = ["date", "peak_flow_cfs"]
    df["date"]          = pd.to_datetime(df["date"], errors="coerce")
    df["peak_flow_cfs"] = pd.to_numeric(df["peak_flow_cfs"], errors="coerce")
    df = df.dropna(subset=["date", "peak_flow_cfs"])
    df["water_year"]    = df["date"].apply(_water_year)

    df = (df.groupby("water_year", as_index=False)["peak_flow_cfs"]
            .max()
            .dropna(subset=["peak_flow_cfs"]))
    return df[(df["water_year"] >= start_wy) & (df["water_year"] <= end_wy)]


def fetch_annual_avg_flow(site_no: str, start_wy: int, end_wy: int) -> pd.DataFrame:
    """
    Pull annual mean discharge from USGS NWIS statistics service.
    Returns DataFrame with columns [water_year, avg_flow_cfs].
    Returns empty DataFrame (silently) if the service has no annual stats.

    Endpoint: waterservices.usgs.gov/nwis/stat/?statReportType=annual
    """
    url = (
        f"https://waterservices.usgs.gov/nwis/stat/"
        f"?sites={site_no}&statReportType=annual&statYearType=water"
        f"&parameterCd=00060&format=rdb"
    )
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
    except Exception:
        return pd.DataFrame(columns=["water_year", "avg_flow_cfs"])

    lines = [l for l in r.text.splitlines()
             if not l.startswith("#") and l.strip()]
    if len(lines) < 3:
        return pd.DataFrame(columns=["water_year", "avg_flow_cfs"])

    header     = lines[0].split("\t")
    data_lines = [l for l in lines[2:]
                  if l.strip()
                  and not l.startswith("5s")
                  and not l.startswith("site_no")]
    if not data_lines:
        return pd.DataFrame(columns=["water_year", "avg_flow_cfs"])

    rows = [l.split("\t") for l in data_lines]
    df   = pd.DataFrame(rows, columns=header[:len(rows[0])])

    wy_col   = next((c for c in df.columns if "year" in c.lower()), None)
    mean_col = next((c for c in df.columns if "mean" in c.lower()), None)
    if wy_col is None or mean_col is None:
        return pd.DataFrame(columns=["water_year", "avg_flow_cfs"])

    df = df[[wy_col, mean_col]].copy()
    df.columns         = ["water_year", "avg_flow_cfs"]
    df["water_year"]   = pd.to_numeric(df["water_year"],   errors="coerce")
    df["avg_flow_cfs"] = pd.to_numeric(df["avg_flow_cfs"], errors="coerce")
    df = df.dropna().copy()
    df["water_year"]   = df["water_year"].astype(int)
    return df[(df["water_year"] >= start_wy) & (df["water_year"] <= end_wy)]


def fetch_station_name(site_no: str) -> str:
    """Return human-readable station name from USGS site service, or fallback string."""
    url = (
        f"https://waterservices.usgs.gov/nwis/site/"
        f"?sites={site_no}&format=rdb&siteOutput=expanded"
    )
    try:
        r = requests.get(url, timeout=15)
        for line in r.text.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) > 2 and parts[0] == site_no:
                return parts[2].strip().title()
    except Exception:
        pass
    return f"USGS Station {site_no}"


# ══════════════════════════════════════════════════════════════════════════════
#  CHART BUILD
# ══════════════════════════════════════════════════════════════════════════════

def build_annual_flow_chart(
    datasets,
    custom_title="",
    show_avg=True,
    organization="WRA, Inc.",
):
    """
    Build Balance Hydrologics-style dual-axis grouped bar chart.
    Supports 1 or 2 stations (side-by-side groups within each year).

    datasets — list of dicts, one per station:
        peak_df      : DataFrame [water_year, peak_flow_cfs]
        avg_df       : DataFrame [water_year, avg_flow_cfs]  (may be empty)
        station_id   : str
        station_name : str
        peak_color   : hex color for peak bars
        avg_color    : hex color for avg bars
        label        : display label (overrides station_name if provided)
    """
    global _afg_fig

    n_stations = len(datasets)

    # Union of all water years across all stations
    all_years = sorted(
        set().union(*[set(d["peak_df"]["water_year"]) for d in datasets])
    )
    n_years = len(all_years)
    x       = np.arange(n_years)

    # Bar geometry — each water year is one group.
    # Within each group: [s0_peak, s0_avg, s1_peak, s1_avg] (if 2 stations + avg)
    # One x-slot per station now — Peak and Average share the same center
    # position instead of sitting side-by-side.
    total_bars  = n_stations
    group_width = 0.72
    slot_w      = group_width / total_bars
    peak_w      = slot_w * 0.45   # narrow, solid, drawn in front
    avg_w       = slot_w * 1.05   # thicker, translucent, drawn behind

    fig, ax1 = plt.subplots(figsize=(16, 7))
    fig.patch.set_facecolor(WRA_WHITE)
    ax1.set_facecolor(WRA_WHITE)
    ax2 = ax1.twinx()

    # twinx() stacks ax2 on top of ax1 by default. We want the opposite —
    # Peak Flow (ax1) drawn in FRONT of Average Flow (ax2) — so flip the
    # stacking order and hide ax1's now-redundant background patch.
    ax1.set_zorder(ax2.get_zorder() + 1)
    ax1.patch.set_visible(False)

    all_peak_vals, all_avg_vals = [], []

    for i_s, d in enumerate(datasets):
        peak_map = dict(zip(d["peak_df"]["water_year"], d["peak_df"]["peak_flow_cfs"]))
        avg_map  = (dict(zip(d["avg_df"]["water_year"],  d["avg_df"]["avg_flow_cfs"]))
                    if show_avg and not d["avg_df"].empty else {})

        peak_vals = [peak_map.get(y, 0) for y in all_years]
        avg_vals  = [avg_map.get(y,  0) for y in all_years]

        all_peak_vals.extend(v for v in peak_vals if v > 0)
        all_avg_vals.extend(v  for v in avg_vals  if v > 0)

        lbl    = d["label"] or d["station_name"] or f"Station {d['station_id']}"
        offset = (i_s - (total_bars - 1) / 2.0) * slot_w

        # Average Flow drawn first, wider and semi-transparent — sits as a
        # "halo" behind the Peak Flow bar at the same water-year position.
        if show_avg and avg_vals and any(v > 0 for v in avg_vals):
            ax2.bar(
                x + offset, avg_vals, width=avg_w,
                color=d["avg_color"], alpha=0.7, zorder=2,
                label=(f"{lbl} — Annual Average Flow (cfs)"
                       if n_stations > 1 else "Annual Average Flow (cfs)"),
            )

        # Peak Flow drawn on top, narrower and fully opaque.
        ax1.bar(
            x + offset, peak_vals, width=peak_w,
            color=d["peak_color"], zorder=4,
            label=(f"{lbl} — Annual Peak Flow (cfs)"
                   if n_stations > 1 else "Annual Peak Flow (cfs)"),
        )

    # ── Axes formatting ───────────────────────────────────────────────────────
    ax1.set_xticks(x)
    ax1.set_xticklabels([str(y) for y in all_years],
                        rotation=90, fontsize=8, fontfamily="Verdana")
    ax1.set_xlim(-0.6, n_years - 0.4)
    ax1.set_ylim(0, max(all_peak_vals, default=1) * 1.1)
    ax2.set_ylim(0, max(all_avg_vals,  default=1) * 1.5 if all_avg_vals else 10)

    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax1.tick_params(axis="y", labelsize=8)
    ax2.tick_params(axis="y", labelsize=8)

    ax1.set_ylabel("Annual Peak Flow (cfs)",    fontsize=10,
                   fontfamily="Verdana", labelpad=6)
    ax2.set_ylabel("Annual Average Flow (cfs)", fontsize=10,
                   fontfamily="Verdana", labelpad=6)
    ax1.set_xlabel("Water Year", fontsize=10,
                   fontfamily="Verdana", labelpad=6)

    ax1.grid(axis="y", color=WRA_GRID, linewidth=0.5, zorder=0)
    ax1.grid(axis="x", visible=False)

    # ── Stats box (top-left) ──────────────────────────────────────────────────
    stats_lines = []
    for d in datasets:
        lbl      = d["label"] or d["station_name"] or f"Station {d['station_id']}"
        peak_map = dict(zip(d["peak_df"]["water_year"], d["peak_df"]["peak_flow_cfs"]))
        avg_map  = (dict(zip(d["avg_df"]["water_year"],  d["avg_df"]["avg_flow_cfs"]))
                    if show_avg and not d["avg_df"].empty else {})
        vp = [v for v in peak_map.values() if v > 0]
        va = [v for v in avg_map.values()  if v > 0]
        wy_s = min(peak_map) if peak_map else "?"
        wy_e = max(peak_map) if peak_map else "?"
        hdr  = f"{lbl}\n" if n_stations > 1 else ""
        avg_line = (f"Mean Annual Average Flow (WY{wy_s}–WY{wy_e}) = "
                    f"{np.mean(va):,.0f} cfs" if va else "")
        block = (f"{hdr}Mean Annual Peak Flow (WY{wy_s}–WY{wy_e}) = "
                 f"{np.mean(vp):,.0f} cfs")
        if avg_line:
            block += f"\n{avg_line}"
        stats_lines.append(block)

    ax1.text(
        0.01, 0.97, "\n\n".join(stats_lines),
        transform=ax1.transAxes, fontsize=8, va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.5", facecolor=WRA_GOLD_FILL,
                  edgecolor=WRA_GOLD, linewidth=1.2),
        fontfamily="Verdana", linespacing=1.5,
    )

    # ── USGS source box (top-right) ───────────────────────────────────────────
    ax1.text(
        0.99, 0.97, "Data collected and provided by the USGS.",
        transform=ax1.transAxes, fontsize=8, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.4", facecolor=WRA_GOLD_FILL,
                  edgecolor=WRA_GOLD, linewidth=1.2),
        fontfamily="Verdana",
    )

    # ── Legend (deduplicated across both axes) ────────────────────────────────
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    seen, uh, ul = set(), [], []
    for h, l in zip(h1 + h2, l1 + l2):
        if l not in seen:
            seen.add(l); uh.append(h); ul.append(l)
    leg = ax1.legend(uh, ul, fontsize=8.5, framealpha=1,
                     edgecolor="#AAAAAA",
                     bbox_to_anchor=(0.99, 0.87), loc="upper right")
    for t in leg.get_texts():
        t.set_fontfamily("Verdana")

    # ── Title ─────────────────────────────────────────────────────────────────
    if custom_title.strip():
        title_str = custom_title.strip()
    else:
        ids_str   = " | ".join(f"USGS {d['station_id']}" for d in datasets)
        names_str = " | ".join(
            d["label"] or d["station_name"] or f"Station {d['station_id']}"
            for d in datasets
        )
        title_str = (
            f"Annual Peak Flow and Average Flow Summary Data\n"
            f"{ids_str} — {names_str}"
        )
    fig.suptitle(title_str, fontsize=12, fontweight="bold",
                 fontfamily="Verdana", y=0.99)

    # ── Footer caption ────────────────────────────────────────────────────────
    last_year = all_years[-1] if all_years else "?"
    captions  = []
    for d in datasets:
        pk_map = dict(zip(d["peak_df"]["water_year"], d["peak_df"]["peak_flow_cfs"]))
        av_map = (dict(zip(d["avg_df"]["water_year"],  d["avg_df"]["avg_flow_cfs"]))
                  if show_avg and not d["avg_df"].empty else {})
        lp  = pk_map.get(last_year, 0)
        la  = av_map.get(last_year, 0)
        lbl = d["label"] or d["station_name"] or f"Station {d['station_id']}"
        captions.append(
            f"{lbl}: WY{last_year} peak = {lp:,.0f} cfs"
            + (f", avg = {la:,.0f} cfs" if la else "")
        )
    fig.text(0.5, 0.005, "   |   ".join(captions),
             ha="center", va="bottom", fontsize=7.5,
             fontfamily="Verdana", color="#444444", style="italic")
    fig.text(0.99, 0.005, f"© {_dt.date.today().year} {organization}",
             ha="right",  va="bottom", fontsize=7,
             fontfamily="Verdana", color="#888888")

    fig.subplots_adjust(left=0.07, right=0.93, top=0.91, bottom=0.15)
    _afg_fig = fig
    plt.show()


# ══════════════════════════════════════════════════════════════════════════════
#  WIDGETS — ipywidgets GUI
# ══════════════════════════════════════════════════════════════════════════════

_afg_fig = None   # most recently generated figure (for SVG save)

# ── Station 1 ─────────────────────────────────────────────────────────────────
afg_s1_id = widgets.Text(
    value="11164500", description="Station ID:",
    placeholder="e.g. 11164500",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="240px"),
)
afg_s1_peak_color = widgets.ColorPicker(
    value=WRA_PEAK_BLUE, description="Peak color:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="175px"),
)
afg_s1_avg_color = widgets.ColorPicker(
    value=WRA_AVG_CYAN, description="Avg color:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="175px"),
)
afg_s1_label = widgets.Text(
    value="", description="Label (opt.):",
    placeholder="e.g. San Francisquito Ck",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

# ── Station 2 (optional) ──────────────────────────────────────────────────────
afg_s2_enable = widgets.Checkbox(
    value=False,
    description="Compare a second station",
    style={"description_width": "initial"},
)
afg_s2_box = widgets.VBox([])

afg_s2_id = widgets.Text(
    value="", description="Station ID:",
    placeholder="e.g. 11143000",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="240px"),
)
afg_s2_peak_color = widgets.ColorPicker(
    value="#2E4F8A", description="Peak color:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="175px"),
)
afg_s2_avg_color = widgets.ColorPicker(
    value="#A8D8F0", description="Avg color:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="175px"),
)
afg_s2_label = widgets.Text(
    value="", description="Label (opt.):",
    placeholder="Station 2 name",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)


def _toggle_s2(change):
    afg_s2_box.children = (
        [widgets.HTML("<b>Station 2</b>"),
         widgets.HBox([afg_s2_id, afg_s2_peak_color,
                       afg_s2_avg_color, afg_s2_label])]
        if change["new"] else []
    )


afg_s2_enable.observe(_toggle_s2, names="value")

# ── Chart options ─────────────────────────────────────────────────────────────
_CURRENT_YEAR = _dt.date.today().year

afg_custom_range_toggle = widgets.Checkbox(
    value=False,
    description="Customize water year range (default: show all available data)",
    style={"description_width": "initial"},
)
afg_start_wy = widgets.BoundedIntText(
    value=1900, min=1900, max=2100, description="Start WY:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="160px"),
)
afg_end_wy = widgets.BoundedIntText(
    value=_CURRENT_YEAR + 1, min=1900, max=2100, description="End WY:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="160px"),
)
afg_range_box = widgets.VBox([])   # populated only when the toggle is checked


def _toggle_range(change):
    afg_range_box.children = (
        [widgets.HBox([afg_start_wy, afg_end_wy])] if change["new"] else []
    )


afg_custom_range_toggle.observe(_toggle_range, names="value")
afg_title_input = widgets.Text(
    value="", description="Title (opt.):",
    placeholder="e.g. San Francisquito Creek at Stanford",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="480px"),
)
afg_org_input = widgets.Text(
    value="WRA, Inc.", description="Organization:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="280px"),
)

# ── Buttons ───────────────────────────────────────────────────────────────────
afg_run_btn  = widgets.Button(
    description="Generate Chart",
    button_style="primary",
    layout=widgets.Layout(width="140px"),
)
afg_save_btn = widgets.Button(
    description="Save as SVG",
    button_style="success",
    layout=widgets.Layout(width="110px"),
)
afg_status = widgets.Label(value="Enter a USGS station ID and click Generate Chart.")
afg_out    = widgets.Output()

# ── UI layout ─────────────────────────────────────────────────────────────────
afg_ui = widgets.VBox([
    widgets.HTML(
        "<div style='display:flex;align-items:center;gap:14px;"
        "background:white;padding:10px 20px;border-radius:6px;"
        f"border:2px solid {WRA_NAVY};margin-bottom:6px'>"
        f"<img src='data:image/png;base64,{_LOGO_B64}' "
        "style='height:42px;width:auto;display:block' alt='WRA logo'>"
        f"<span style='color:{WRA_NAVY};font-size:16px;"
        "font-family:Verdana;font-weight:bold'>"
        "Annual Peak Flow &amp; Average Flow Chart Generator"
        "</span></div>"
    ),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HTML("<b>Station 1</b>"),
    widgets.HBox([afg_s1_id, afg_s1_peak_color, afg_s1_avg_color, afg_s1_label]),
    afg_s2_enable,
    afg_s2_box,
    widgets.HTML("<hr style='margin:4px 0'>"),
    afg_custom_range_toggle,
    afg_range_box,
    widgets.HBox([afg_title_input]),
    widgets.HBox([afg_org_input]),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([afg_run_btn, afg_save_btn, afg_status]),
    afg_out,
])


# ── Callback: Generate Chart ──────────────────────────────────────────────────
def _on_afg_run(b):
    with afg_out:
        clear_output(wait=True)
        afg_status.value = "Fetching data from USGS…"
        try:
            if afg_custom_range_toggle.value:
                start_wy = int(afg_start_wy.value)
                end_wy   = int(afg_end_wy.value)
            else:
                # No custom range selected -> pull the full available record
                start_wy = 1900
                end_wy   = _CURRENT_YEAR + 1

            # ── Station 1 ─────────────────────────────────────────────────────
            sid1 = afg_s1_id.value.strip()
            if not sid1:
                afg_status.value = "Please enter a Station 1 ID."
                return

            afg_status.value = f"Fetching peak flows for Station {sid1}…"
            peak1 = fetch_peak_flows(sid1, start_wy, end_wy)
            if peak1.empty:
                afg_status.value = (
                    f"No peak flow data found for Station {sid1} "
                    f"in WY{start_wy}–{end_wy}. "
                    "Check the station ID and year range."
                )
                return

            afg_status.value = f"Fetching annual average flows for Station {sid1}…"
            avg1  = fetch_annual_avg_flow(sid1, start_wy, end_wy)
            name1 = fetch_station_name(sid1)

            datasets = [{
                "peak_df":      peak1,
                "avg_df":       avg1,
                "station_id":   sid1,
                "station_name": name1,
                "peak_color":   afg_s1_peak_color.value,
                "avg_color":    afg_s1_avg_color.value,
                "label":        afg_s1_label.value.strip(),
            }]

            # ── Station 2 (optional) ──────────────────────────────────────────
            if afg_s2_enable.value:
                sid2 = afg_s2_id.value.strip()
                if sid2:
                    afg_status.value = f"Fetching peak flows for Station {sid2}…"
                    peak2 = fetch_peak_flows(sid2, start_wy, end_wy)
                    afg_status.value = f"Fetching annual average flows for Station {sid2}…"
                    avg2  = fetch_annual_avg_flow(sid2, start_wy, end_wy)
                    name2 = fetch_station_name(sid2)
                    datasets.append({
                        "peak_df":      peak2,
                        "avg_df":       avg2,
                        "station_id":   sid2,
                        "station_name": name2,
                        "peak_color":   afg_s2_peak_color.value,
                        "avg_color":    afg_s2_avg_color.value,
                        "label":        afg_s2_label.value.strip(),
                    })

            afg_status.value = "Rendering chart…"
            build_annual_flow_chart(
                datasets,
                custom_title=afg_title_input.value,
                show_avg=True,
                organization=afg_org_input.value.strip() or "WRA, Inc.",
            )

            # ── Summary stats printed below chart ─────────────────────────────
            print("\n" + "=" * 70)
            print("  ANNUAL FLOW SUMMARY")
            print("=" * 70)
            for d in datasets:
                lbl = d["label"] or d["station_name"] or f"Station {d['station_id']}"
                pk  = d["peak_df"]
                av  = d["avg_df"]
                print(f"\n  {lbl}  (USGS {d['station_id']})")
                print(f"    Record span    : WY{pk['water_year'].min()} – "
                      f"WY{pk['water_year'].max()}")
                print(f"    N years (peak) : {len(pk)}")
                print(f"    Mean peak flow : {pk['peak_flow_cfs'].mean():,.0f} cfs")
                print(f"    Max peak flow  : {pk['peak_flow_cfs'].max():,.0f} cfs  "
                      f"(WY{int(pk.loc[pk['peak_flow_cfs'].idxmax(), 'water_year'])})")
                if not av.empty:
                    print(f"    Mean avg flow  : {av['avg_flow_cfs'].mean():,.1f} cfs")
                else:
                    print("    Avg flow data  : not available for this station")
            print()

            n_str = " | ".join(
                f"{d['label'] or d['station_name']}: n = {len(d['peak_df'])} yrs"
                for d in datasets
            )
            afg_status.value = f"Done.  {n_str}"

        except requests.exceptions.HTTPError as e:
            afg_status.value = f"HTTP error: {e}"
        except RuntimeError as e:
            afg_status.value = str(e)
        except Exception as e:
            afg_status.value = f"Unexpected error: {e}"
            raise


# ── Callback: Save SVG ────────────────────────────────────────────────────────
def _on_afg_save(b):
    global _afg_fig
    if _afg_fig is None:
        afg_status.value = "Generate a chart first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save annual flow chart as SVG",
            defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            _afg_fig.savefig(path, format="svg", bbox_inches="tight")
            afg_status.value = f"Saved: {path}"
        else:
            afg_status.value = "Save cancelled."
    except Exception as e:
        afg_status.value = f"Save error: {e}"


afg_run_btn.on_click(_on_afg_run)
afg_save_btn.on_click(_on_afg_save)

# ── Launch ────────────────────────────────────────────────────────────────────
display(afg_ui)

# ══════════════════════════════════════════════════════════════════════════════
# ENDPOINT NOTES
# ══════════════════════════════════════════════════════════════════════════════
# Peak flows:
#   Primary:  https://nwis.waterdata.usgs.gov/usa/nwis/peak?site_no=...&format=rdb
#   Fallback: https://waterdata.usgs.gov/nwis/peak?site_no=...&format=rdb
#   The old waterservices.usgs.gov/nwis/peak/ endpoint returns 404 for most
#   stations because peak flows were never served from that host — use the
#   nwis.waterdata.usgs.gov host above.
#
# Annual average flows:
#   https://waterservices.usgs.gov/nwis/stat/?statReportType=annual&statYearType=water
#   Returns empty gracefully if the station has no annual stats computed.
#
# Station name:
#   https://waterservices.usgs.gov/nwis/site/?sites=...&format=rdb&siteOutput=expanded



## change to mean and max peak flow

---

## Log Pearson III Calculator


In [3]:
_LOGO_B64 = "..."

# ═══════════════════════════════════════════════════════════════════════════════
#  Log-Pearson Type III Flood Frequency Analysis — ipywidgets GUI
#  Bulletin 17C / standard LP3 method
#  Produces probability plot matching USGS Peakfq / Figure 10-13 style
# ═══════════════════════════════════════════════════════════════════════════════
#
#  KEY IMPLEMENTATION NOTES:
#  • Quantiles use Wilson-Hilferty (WH) approximation — same as USGS Peakfq.
#    This avoids the bounded-support artifact from scipy.pearson3.isf() which
#    produces a spurious linear section at high AEP for negative-skew records.
#  • Variance of estimate uses Kite (1977) / Bulletin 17C approximation,
#    appropriate for systematic records without historical or censored data.
#  • For sites with PILFs or historical data, full EMA (Cohn et al. 2001)
#    would give slightly different variance values — implement via PeakFQ.
#  • Confidence limits use z (standard normal) per B17C eq. 7-31.
#
#  DEPENDENCIES:  pandas, numpy, scipy, matplotlib, requests, ipywidgets
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import rcParams
from scipy import stats
from scipy.stats import norm
import requests
from io import StringIO
import ipywidgets as widgets
from IPython.display import display, clear_output
import tkinter as tk
from tkinter import filedialog

rcParams["font.family"]     = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"]  = 13
rcParams["axes.labelsize"]  = 11
rcParams["xtick.labelsize"] = 9
rcParams["ytick.labelsize"] = 9

# ══════════════════════════════════════════════════════════════════════════════
#  LP3 MATH — Wilson-Hilferty approximation (matches USGS Peakfq)
# ══════════════════════════════════════════════════════════════════════════════

def lp3_fit(peak_va_series):
    """
    Fit Log-Pearson Type III to annual peak flows.
    Returns dict: mean_log, std_log, skew_log (sample), n, logq.
    Uses sample statistics (unbiased) per Bulletin 17C.
    """
    q    = peak_va_series.dropna().values
    q    = q[q > 0]
    logq = np.log10(q)
    n    = len(logq)
    mean = np.mean(logq)
    std  = np.std(logq, ddof=1)
    skew = (n / ((n - 1) * (n - 2))) * np.sum(((logq - mean) / std) ** 3)
    return {"mean_log": mean, "std_log": std, "skew_log": skew,
            "n": n, "logq": logq}


def wh_K(z, g):
    """
    Wilson-Hilferty frequency factor K for skew g and standard normal deviate z.
    This is the same approximation used by USGS Peakfq / Bulletin 17C.
    K maps z → the LP3 frequency factor so that log(Q_p) = mean + K*std.
    For g ≈ 0, reduces to K = z (normal distribution).
    """
    if abs(g) < 1e-6:
        return z
    return (2.0 / g) * ((1.0 + g * z / 6.0 - g**2 / 36.0)**3 - 1.0)


def lp3_quantile(params, aep):
    """
    LP3 quantile at given AEP using Wilson-Hilferty approximation.
    aep in (0,1). Returns discharge in cfs.
    Uses ppf of 1-aep (survival function equivalent) so small AEP → large flow.
    """
    z    = norm.ppf(1.0 - aep)          # z > 0 for rare events (small AEP)
    K    = wh_K(z, params["skew_log"])
    logq = params["mean_log"] + K * params["std_log"]
    return 10.0 ** logq


def lp3_variance_and_ci(params, aep, alpha=0.05):
    """
    Variance of log-quantile estimate and 2.5%/97.5% confidence limits.

    Variance formula: Kite (1977), Bulletin 17C approximation.
      Var(log Q_p) = s² × [(1 + K·g/6 + K²·(1 - g²/4)) / n  +  K² / (2·(n-1))]
    where K is the WH frequency factor.

    Confidence limits use z (standard normal) per B17C eq. 7-31:
      CI = 10^(log Q_p ± z_{1-α/2} × sqrt(Var))
    This is appropriate for large-sample systematic records.

    Returns: (variance_of_log_estimate, lower_cfs, upper_cfs)
    """
    n    = params["n"]
    g    = params["skew_log"]
    s    = params["std_log"]
    z    = norm.ppf(1.0 - aep)
    K    = wh_K(z, g)
    logq = params["mean_log"] + K * s

    # Kite (1977) / B17C variance of log-quantile
    a       = 1.0 + K * g / 6.0 + K**2 * (1.0 - g**2 / 4.0)
    var_logq = s**2 * (abs(a) / n + K**2 / (2.0 * (n - 1)))

    z_crit = norm.ppf(1.0 - alpha / 2.0)    # 1.96 for 95% CI
    se     = np.sqrt(var_logq)
    lower  = 10.0 ** (logq - z_crit * se)
    upper  = 10.0 ** (logq + z_crit * se)
    return var_logq, lower, upper


def plotting_positions(logq_sorted, n):
    """Weibull plotting positions: P(X >= x_i) = rank / (n+1), returns in (0,1)."""
    ranks = np.arange(1, n + 1)
    return ranks / (n + 1)


# ══════════════════════════════════════════════════════════════════════════════
#  DATA FETCH
# ══════════════════════════════════════════════════════════════════════════════

def _hydro_year(dt):
    return dt.year + 1 if dt.month >= 10 else dt.year

def _fetch_usgs_peaks(station_id):
    """Pull USGS peak-flow RDB, return tidy DataFrame."""
    urls = [
        f"https://nwis.waterdata.usgs.gov/usa/nwis/peak?site_no={station_id}&agency_cd=USGS&format=rdb",
        f"https://waterdata.usgs.gov/nwis/peak?site_no={station_id}&agency_cd=USGS&format=rdb",
    ]
    resp = None
    for url in urls:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        if "peak_dt" in r.text or "peak_va" in r.text:
            resp = r
            break
    if resp is None:
        raise ValueError(f"No RDB data for station {station_id}.")

    lines = [l for l in resp.text.splitlines() if l.strip() and not l.startswith("#")]
    if len(lines) < 3:
        raise ValueError(f"No data returned for station {station_id}.")

    header   = lines[0]
    n_cols   = len(header.split("\t"))
    records  = lines[2:]   # skip header and format row
    good     = [l for l in records if len(l.split("\t")) == n_cols]
    if not good:
        raise ValueError(f"Station {station_id}: no data rows matched header.")

    df = pd.read_csv(StringIO("\n".join([header] + good)), sep="\t", low_memory=False)
    if "peak_va" not in df.columns or "peak_dt" not in df.columns:
        raise ValueError(f"Columns 'peak_dt'/'peak_va' not found for {station_id}.")

    df = df[["peak_dt", "peak_va"]].dropna(subset=["peak_va"])
    df["peak_va"] = pd.to_numeric(df["peak_va"], errors="coerce")
    df["peak_dt"] = pd.to_datetime(df["peak_dt"], errors="coerce")
    df = df.dropna(subset=["peak_va", "peak_dt"])
    df["year"] = df["peak_dt"].apply(_hydro_year)
    return df.groupby("year", as_index=False)["peak_va"].max().dropna(subset=["peak_va"])

# ══════════════════════════════════════════════════════════════════════════════
#  PROBABILITY AXIS HELPERS
# ══════════════════════════════════════════════════════════════════════════════

# AEP ticks for the bottom axis (%, same as Peakfq / B17C Figure 10-13)
AEP_TICKS_PCT = [99.5, 99, 95, 90, 80, 65, 50, 35, 20, 10, 5, 4, 2, 1,
                 0.5, 0.2, 0.1]

def aep_to_normal(aep_pct):
    """AEP% → standard-normal deviate (for linear probability axis)."""
    p = np.clip(np.asarray(aep_pct, dtype=float) / 100.0, 1e-9, 1 - 1e-9)
    return norm.ppf(p)

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN PLOT FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def make_lp3_plot(datasets, custom_title="", show_ci=True, rp_list=None):
    """
    datasets: list of dicts {peak_va, station_id, label, color}
    rp_list: return periods (yr) to tabulate
    Returns: dict label → list of row dicts
    """
    global _lp3_fig

    if rp_list is None:
        rp_list = [2, 5, 10, 25, 50, 100, 200, 500, 1000]

    # AEP fine grid for drawing the fitted curve and CI bands.
    # Range: 99.5% → 0.1% (stays well within valid WH range for any skew).
    aep_fine_pct = np.concatenate([
        np.linspace(99.5, 95, 50),
        np.linspace(95, 10, 200),
        np.linspace(10, 0.1, 250),
    ])
    aep_fine     = aep_fine_pct / 100.0
    x_fine       = aep_to_normal(aep_fine_pct)

    fig, ax = plt.subplots(figsize=(14, 8))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    all_summary = {}

    for d in datasets:
        params = lp3_fit(d["peak_va"])
        color  = d["color"]
        label  = d["label"] or f"Station {d['station_id']}"

        # ── Fitted LP3 curve (WH) ─────────────────────────────────────────────
        q_curve = np.array([lp3_quantile(params, a) for a in aep_fine])
        ax.semilogy(x_fine, q_curve, color=color, linewidth=2.0,
                    label=f"{label} — LP3 fit", zorder=4)

        # ── Confidence bands ─────────────────────────────────────────────────
        if show_ci:
            ci_lo = np.array([lp3_variance_and_ci(params, a)[1] for a in aep_fine])
            ci_hi = np.array([lp3_variance_and_ci(params, a)[2] for a in aep_fine])
            ax.semilogy(x_fine, ci_lo, color=color, linewidth=1.0,
                        linestyle="--", alpha=0.7,
                        label=f"{label} — Lower 2.5% confidence limit")
            ax.semilogy(x_fine, ci_hi, color=color, linewidth=1.0,
                        linestyle="--", alpha=0.7,
                        label=f"{label} — Upper 97.5% confidence limit")
            ax.fill_between(x_fine, ci_lo, ci_hi, color=color, alpha=0.08)

        # ── Observed peaks (Weibull plotting positions) ───────────────────────
        logq_sorted = np.sort(params["logq"])[::-1]   # descending by flow
        obs_aep     = plotting_positions(logq_sorted, params["n"])
        obs_x       = aep_to_normal(obs_aep * 100.0)
        obs_q       = 10.0 ** logq_sorted
        ax.semilogy(obs_x, obs_q, marker="o", linestyle="none",
                    color="black", markerfacecolor="white",
                    markersize=5, markeredgewidth=0.8,
                    label=f"{label} — annual peak", zorder=5)

        # ── Summary table rows ────────────────────────────────────────────────
        rows = []
        for rp in rp_list:
            aep_rp       = 1.0 / rp
            q_design     = lp3_quantile(params, aep_rp)
            var_est, lo, hi = lp3_variance_and_ci(params, aep_rp)
            rows.append({
                "Return Period":                f"{rp}-yr",
                "AEP (%)":                      f"{aep_rp*100:.4g}",
                "EMA Estimate (cfs)":           f"{q_design:,.0f}",
                "Variance of Estimate":         f"{var_est:.5f}",
                "Lower 2.5% Confidence Limit":  f"{lo:,.0f}",
                "Upper 97.5% Confidence Limit": f"{hi:,.0f}",
            })
        all_summary[label] = rows

    # ── Bottom x-axis: AEP% (normal probability scale) ───────────────────────
    x_ticks = aep_to_normal(AEP_TICKS_PCT)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([str(v) for v in AEP_TICKS_PCT],
                       fontsize=8.5, fontfamily="Verdana")
    ax.set_xlim(aep_to_normal(99.5), aep_to_normal(0.1))
    ax.set_xlabel("Annual Exceedance Probability, in percent",
                  fontsize=11, fontfamily="Verdana")

    # ── Left y-axis ───────────────────────────────────────────────────────────
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{int(y):,}"))
    ax.yaxis.set_minor_formatter(ticker.NullFormatter())
    ax.set_ylabel("Annual Peak Discharge (cfs)", fontsize=11, fontfamily="Verdana")

    # ── Top x-axis: return period ─────────────────────────────────────────────
    ax2 = ax.twiny()
    rp_top     = [1.01, 1.05, 1.11, 1.25, 2, 5, 10, 25, 50, 100, 200, 500, 1000]
    rp_aep_pct = [100.0 / r for r in rp_top]
    # Only keep ticks within plot range (0.1% – 99.5%)
    rp_top     = [r for r, a in zip(rp_top, rp_aep_pct) if 0.1 <= a <= 99.5]
    rp_aep_pct = [100.0 / r for r in rp_top]
    rp_x       = aep_to_normal(rp_aep_pct)
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(rp_x)
    ax2.set_xticklabels([str(r) if r >= 1 else f"{r:.2f}" for r in rp_top],
                        fontsize=8, fontfamily="Verdana")
    ax2.set_xlabel("Return Period (years)", fontsize=10, fontfamily="Verdana", labelpad=4)

    # ── Grid, title, legend ───────────────────────────────────────────────────
    ax.grid(True, which="both", linestyle=":", linewidth=0.5, alpha=0.6)

    title_str = (custom_title.strip() if custom_title.strip()
                 else "Log-Pearson Type III Flood Frequency Analysis\n" +
                      " | ".join(d["label"] or f"Station {d['station_id']}"
                                 for d in datasets))
    ax.set_title(title_str, fontsize=13, fontweight="bold",
                 fontfamily="Verdana", pad=28)

    handles, labels_leg = ax.get_legend_handles_labels()
    seen, h2, l2 = set(), [], []
    for h, l in zip(handles, labels_leg):
        if l not in seen:
            seen.add(l)
            h2.append(h)
            l2.append(l)
    leg = ax.legend(h2, l2, fontsize=9.5, framealpha=0.9, loc="upper left")
    for t in leg.get_texts():
        t.set_fontfamily("Verdana")

    meta = "  |  ".join(
        f"n = {lp3_fit(d['peak_va'])['n']} yrs  •  "
        f"skew = {lp3_fit(d['peak_va'])['skew_log']:.3f}"
        for d in datasets
    )
    fig.text(0.5, 0.01, meta, ha="center", va="bottom",
             fontsize=8, fontfamily="Verdana", color="#555555")

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    _lp3_fig = fig
    plt.show()

    return all_summary


# ══════════════════════════════════════════════════════════════════════════════
#  WIDGETS
# ══════════════════════════════════════════════════════════════════════════════

_lp3_fig     = None
_lp3_summary = None

lp3_s1_header = widgets.HTML("<b>Station 1</b>")
lp3_s1_id     = widgets.Text(description="Station ID:",
                              placeholder="e.g. 11113000",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="240px"))
lp3_s1_color  = widgets.ColorPicker(value="#c0392b", description="Color:",
                                    style={"description_width": "50px"},
                                    layout=widgets.Layout(width="160px"))
lp3_s1_label  = widgets.Text(value="", description="Label (opt):",
                              placeholder="Station 1 name",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="300px"))

lp3_s2_enable = widgets.Checkbox(value=False,
                                  description="Compare a second station",
                                  style={"description_width": "initial"})
lp3_s2_box    = widgets.VBox([])
lp3_s2_id     = widgets.Text(value="", description="Station ID:",
                              placeholder="e.g. 11109000",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="240px"))
lp3_s2_color  = widgets.ColorPicker(value="#2980b9", description="Color:",
                                    style={"description_width": "50px"},
                                    layout=widgets.Layout(width="160px"))
lp3_s2_label  = widgets.Text(value="", description="Label (opt):",
                              placeholder="Station 2 name",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="300px"))

def _toggle_lp3_s2(change):
    lp3_s2_box.children = (
        [widgets.HTML("<b>Station 2</b>"),
         widgets.HBox([lp3_s2_id, lp3_s2_color, lp3_s2_label])]
        if change["new"] else []
    )
lp3_s2_enable.observe(_toggle_lp3_s2, names="value")

lp3_title_input = widgets.Text(
    value="", description="Title (opt.):",
    placeholder="e.g. Arkansas R. at Pueblo State Park",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="500px"),
)
lp3_ci_toggle = widgets.Checkbox(
    value=True,
    description="Show 2.5%/97.5% confidence limits",
    style={"description_width": "initial"},
)

# Return-period checkboxes
# Standard: 2, 5, 10, 25, 50, 100
# Extrapolated: 200, 500, 1000
_rp_opts_std = [2, 5, 10, 25, 50, 100]
_rp_opts_ext = [200, 500, 1000]
_rp_opts     = _rp_opts_std + _rp_opts_ext

lp3_rp_checks = widgets.VBox([
    widgets.HTML("<b>Design return periods to tabulate:</b>"),
    widgets.HBox([
        widgets.Checkbox(value=True, description=f"{rp}-yr",
                         layout=widgets.Layout(width="90px"))
        for rp in _rp_opts_std
    ]),
    widgets.HBox(
        [widgets.HTML("<span style='color:#888;font-size:11px;margin-right:6px'>"
                      "&#9888; Extrapolations (use with caution):</span>")] +
        [widgets.Checkbox(value=False, description=f"{rp}-yr",
                          layout=widgets.Layout(width="90px"))
         for rp in _rp_opts_ext]
    ),
])

lp3_run_btn      = widgets.Button(description="Run LP3",        button_style="primary",
                                   layout=widgets.Layout(width="100px"))
lp3_save_btn     = widgets.Button(description="Save Plot SVG",  button_style="success",
                                   layout=widgets.Layout(width="120px"))
lp3_save_tbl_btn = widgets.Button(description="Save Table SVG", button_style="info",
                                   layout=widgets.Layout(width="120px"))
lp3_status = widgets.Label(value="")
lp3_out    = widgets.Output()

lp3_ui = widgets.VBox([
    widgets.HTML(
        "<div style='display:flex;align-items:center;gap:14px;"
        "background:white;padding:10px 20px;border-radius:6px;"
        "border:2px solid #003B5C;margin-bottom:6px'>"
        "<img src='data:image/png;base64," + _LOGO_B64 + "' "
        "style='height:44px;width:auto'>"
        "<span style='color:#003B5C;font-size:16px;font-family:Verdana;font-weight:bold'>"
        "Log-Pearson Type III Flood Frequency Analysis</span></div>"
    ),
    widgets.HTML("<hr style='margin:4px 0'>"),
    lp3_s1_header,
    widgets.HBox([lp3_s1_id, lp3_s1_color, lp3_s1_label]),
    lp3_s2_enable,
    lp3_s2_box,
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([lp3_title_input, lp3_ci_toggle]),
    lp3_rp_checks,
    widgets.HBox([lp3_run_btn, lp3_save_btn, lp3_save_tbl_btn, lp3_status]),
    lp3_out,
])

# ── Callbacks ─────────────────────────────────────────────────────────────────

def _on_lp3_run(b):
    with lp3_out:
        clear_output(wait=True)
        lp3_status.value = "Fetching data..."
        try:
            datasets = []
            df1 = _fetch_usgs_peaks(lp3_s1_id.value.strip())
            datasets.append({
                "peak_va":    df1["peak_va"],
                "station_id": lp3_s1_id.value.strip(),
                "label":      lp3_s1_label.value.strip(),
                "color":      lp3_s1_color.value,
            })
            if lp3_s2_enable.value and lp3_s2_id.value.strip():
                df2 = _fetch_usgs_peaks(lp3_s2_id.value.strip())
                datasets.append({
                    "peak_va":    df2["peak_va"],
                    "station_id": lp3_s2_id.value.strip(),
                    "label":      lp3_s2_label.value.strip(),
                    "color":      lp3_s2_color.value,
                })

            std_cbs      = lp3_rp_checks.children[1].children
            ext_cbs      = lp3_rp_checks.children[2].children[1:]
            selected_rps = [rp for rp, cb in zip(_rp_opts, list(std_cbs)+list(ext_cbs)) if cb.value]
            if not selected_rps:
                selected_rps = _rp_opts_std

            ns = " | ".join(f"{d['label'] or d['station_id']}: n={len(d['peak_va'])}"
                            for d in datasets)
            lp3_status.value = f"{ns} — fitting LP3..."

            global _lp3_summary
            _lp3_summary = make_lp3_plot(
                datasets,
                custom_title=lp3_title_input.value,
                show_ci=lp3_ci_toggle.value,
                rp_list=selected_rps,
            )

            # LP3 parameters
            print("\n" + "=" * 70)
            print("  LOG-PEARSON TYPE III PARAMETERS (Bulletin 17C / WH)")
            print("=" * 70)
            for d in datasets:
                p    = lp3_fit(d["peak_va"])
                name = d["label"] or f"Station {d['station_id']}"
                print(f"\n  {name}")
                print(f"    n             = {p['n']}")
                print(f"    Mean (log10)  = {p['mean_log']:.4f}")
                print(f"    Std  (log10)  = {p['std_log']:.4f}")
                print(f"    Skew (log10)  = {p['skew_log']:.4f}")
            print()

            # Design flow tables — one per station, station name as title
            print("=" * 70)
            print("  DESIGN FLOWS — LP3 with 2.5%/97.5% Confidence Limits")
            print("=" * 70)
            for station_title, rows in _lp3_summary.items():
                df_tbl = pd.DataFrame(rows)
                print(f"\n  {station_title}")
                print("-" * 70)
                display(df_tbl)

            lp3_status.value = "Done."

        except Exception as e:
            lp3_status.value = f"Error: {e}"
            raise


def _on_lp3_save(b):
    global _lp3_fig
    if _lp3_fig is None:
        lp3_status.value = "Run LP3 first."
        return
    try:
        root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save LP3 figure as SVG", defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            _lp3_fig.savefig(path, format="svg", bbox_inches="tight")
            lp3_status.value = f"Saved: {path}"
        else:
            lp3_status.value = "Save cancelled."
    except Exception as e:
        lp3_status.value = f"Save error: {e}"


def _on_lp3_save_table(b):
    """Save each station's design-flow table as SVG, stacked in one figure."""
    global _lp3_summary
    if _lp3_summary is None:
        lp3_status.value = "Run LP3 first."
        return
    try:
        root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save design-flow table as SVG", defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if not path:
            lp3_status.value = "Save cancelled."
            return

        fig_parts  = [(title, pd.DataFrame(rows)) for title, rows in _lp3_summary.items()]
        n_stations = len(fig_parts)
        n_cols     = len(fig_parts[0][1].columns)
        total_rows = sum(len(df) for _, df in fig_parts)
        fig_h      = max(3.0, 0.42 * (total_rows + 3 * n_stations + 2))
        fig_w      = max(12, 2.2 * n_cols)

        fig, axes = plt.subplots(n_stations, 1, figsize=(fig_w, fig_h), squeeze=False)
        fig.patch.set_facecolor("white")

        for ax_row, (station_title, df_tbl) in zip(axes, fig_parts):
            ax = ax_row[0]
            ax.axis("off")
            n_r, n_c = len(df_tbl), len(df_tbl.columns)

            def _rp_val(s):
                try: return float(str(s).replace("-yr","").replace(",",""))
                except: return 0

            row_colors = [
                ["#fff8e1" if _rp_val(row.get("Return Period","0")) > 100 else "#ffffff"] * n_c
                for _, row in df_tbl.iterrows()
            ]

            tbl = ax.table(
                cellText=df_tbl.values, colLabels=df_tbl.columns.tolist(),
                cellColours=row_colors, colColours=["#d0e4f7"] * n_c,
                cellLoc="center", loc="center",
            )
            tbl.auto_set_font_size(False)
            tbl.set_fontsize(9)
            tbl.auto_set_column_width(col=list(range(n_c)))
            for (r, c), cell in tbl.get_celld().items():
                cell.set_edgecolor("#cccccc")
                cell.set_text_props(
                    fontweight="bold" if r == 0 else "normal",
                    fontfamily="Verdana",
                )
            ax.set_title(station_title, fontsize=11, fontweight="bold",
                         fontfamily="Verdana", pad=6, loc="left")

        fig.text(0.5, 0.005,
                 "Rows with RP > 100-yr (yellow) are extrapolations beyond the period of record.",
                 ha="center", va="bottom", fontsize=7.5, color="#888888", fontfamily="Verdana")
        plt.tight_layout(rect=[0, 0.025, 1, 1])
        fig.savefig(path, format="svg", bbox_inches="tight")
        plt.close(fig)
        lp3_status.value = f"Table saved: {path}"

    except Exception as e:
        lp3_status.value = f"Table save error: {e}"


lp3_run_btn.on_click(_on_lp3_run)
lp3_save_btn.on_click(_on_lp3_save)
lp3_save_tbl_btn.on_click(_on_lp3_save_table)

# ── Launch ────────────────────────────────────────────────────────────────────
display(lp3_ui)

# ══════════════════════════════════════════════════════════════════════════════
# IMPLEMENTATION NOTES
# ══════════════════════════════════════════════════════════════════════════════
# Quantile method:
#   • Wilson-Hilferty (WH) approximation — same as USGS Peakfq software.
#   • K = (2/g)*[(1 + g*z/6 - g²/36)³ - 1], where z = norm.ppf(1-AEP)
#   • log(Q_p) = mean_log + K * std_log
#   • Does NOT hit the bounded-support artifact of scipy.pearson3.isf()
#     which caused the spurious linear section for negative-skew records.
#
# Variance / CI method:
#   • Kite (1977) formula per Bulletin 17C, appropriate for systematic records.
#   • For sites with PILFs, historical data, or censored data, full EMA
#     (Cohn et al. 2001) gives different variance values — use USGS PeakFQ.
#   • Confidence limits use z (standard normal) per B17C eq. 7-31.
#
# AEP grid:
#   • Range 99.5% → 0.1% (1000-yr). Avoids extremes where WH degrades.
#   • Plotting positions: Weibull (rank/(n+1))

---
---

## Daily Flow and Range

In [4]:
# =============================================================================
#  WRA Historical Daily Flow Range Viewer — ipywidgets version (for JupyterHub)
#  Renders entirely inline below the cell. No Tkinter, no separate windows.
#  Shows the historical min-max band + mean daily flow for one or two USGS
#  gauges. Water-year range is optional — leave unchecked to use all
#  available data, or check it to specify a custom range.
# =============================================================================

import datetime as dt
import calendar

import tkinter as tk
from tkinter import filedialog

import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import rcParams

import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

_LOGO_B64 = "..."


# -- WRA brand ----------------------------------------------------------------
WRA_NAVY = "#7D3B2E"
WRA_GOLD = "#C9A84C"

rcParams["font.family"] = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"] = 13
rcParams["axes.labelsize"] = 11
rcParams["xtick.labelsize"] = 8
rcParams["ytick.labelsize"] = 8

# -- Helper functions -----------------------------------------------------------
LINESTYLE_OPTIONS = [("Solid", "-"), ("Dashed", "--"), ("Dash-dot", "-."), ("Dotted", ":")]


def water_year_of(date):
    return date.year + 1 if date.month >= 10 else date.year

def to_plot_date(date, target_wy):
    ref_year = target_wy - 1 if date.month >= 10 else target_wy
    try:
        return dt.date(ref_year, date.month, date.day)
    except ValueError:
        return dt.date(ref_year, 2, 28)

def fetch_site_daily_flow(site_no, start, end=None):
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "json",
        "sites": site_no,
        "startDT": start,
        "parameterCd": "00060",
        "statCd": "00003",
    }
    if end:
        params["endDT"] = end
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    values = data["value"]["timeSeries"][0]["values"][0]["value"]
    df = pd.DataFrame(values)
    if df.empty:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
    df["flow_cfs"] = pd.to_numeric(df["value"])
    return df.rename(columns={"dateTime": "date"})[["date", "flow_cfs"]]

def process_site(df_all, target_wy):
    df_all = df_all.copy()
    df_all["water_year"] = df_all["date"].apply(water_year_of)
    df_all["plot_date"] = df_all["date"].apply(lambda d: to_plot_date(d, target_wy))
    stats = df_all.groupby("plot_date")["flow_cfs"].agg(["min", "mean", "max"]).reset_index()
    stats["plot_date"] = pd.to_datetime(stats["plot_date"])
    stats = stats.sort_values("plot_date")
    return stats


def build_monthly_table(raw_df):
    """Monthly average/maximum/minimum daily flow, computed across every
    year in the fetched record (calendar-month grouping, Jan-Dec)."""
    d = raw_df.copy()
    d["month"] = d["date"].dt.month
    g = d.groupby("month")["flow_cfs"].agg(["mean", "max", "min"])
    g = g.reindex(range(1, 13))
    g.index = [calendar.month_abbr[m] for m in g.index]
    g.index.name = "Month"
    g.columns = ["Average (cfs)", "Maximum (cfs)", "Minimum (cfs)"]
    return g.round(1)


def fetch_period_of_record(site_no, parameter_cd="00060"):
    """Full available period of record for a site/parameter, from the NWIS
    site service's series catalog (a lightweight metadata call, not a full
    daily-values pull)."""
    url = "https://waterservices.usgs.gov/nwis/site/"
    params = {"sites": site_no, "format": "rdb",
              "seriesCatalogOutput": "true", "parameterCd": parameter_cd}
    try:
        r = requests.get(url, params=params, timeout=15)
        lines = [l for l in r.text.splitlines() if not l.startswith("#") and l.strip()]
        if len(lines) < 3:
            return None, None
        header = lines[0].split("\t")
        rows = [l.split("\t") for l in lines[2:] if l.strip()]
        if not rows:
            return None, None
        df = pd.DataFrame(rows, columns=header[: len(rows[0])])
        if "begin_date" not in df.columns or "end_date" not in df.columns:
            return None, None
        begin = pd.to_datetime(df["begin_date"], errors="coerce").min()
        end = pd.to_datetime(df["end_date"], errors="coerce").max()
        if pd.isna(begin) or pd.isna(end):
            return None, None
        return begin.date(), end.date()
    except Exception:
        return None, None


def build_summary_table(raw_df, total_start, total_end):
    """The monthly table with two extra rows on top: the full period of
    record available for the site, and the period actually used here."""
    monthly = build_monthly_table(raw_df)
    used_start, used_end = raw_df["date"].min().date(), raw_df["date"].max().date()
    total_txt = (f"{total_start} to {total_end}" if total_start else "Unknown")
    info = pd.DataFrame(
        {"Average (cfs)": [total_txt, f"{used_start} to {used_end}"],
         "Maximum (cfs)": ["—", "—"],
         "Minimum (cfs)": ["—", "—"]},
        index=["Total period available", "Period used"],
    )
    info.index.name = "Month"
    return pd.concat([info, monthly])


def build_monthly_avg_series(raw_df, target_wy):
    """Monthly-average flow, one point per month, placed on the same
    plot-date axis as everything else (mid-month, day 15)."""
    d = raw_df.copy()
    d["month"] = d["date"].dt.month
    monthly = d.groupby("month")["flow_cfs"].mean().reindex(range(1, 13))
    rows = []
    for m in range(1, 13):
        mid_date = dt.date(2001, m, 15)
        rows.append({"plot_date": to_plot_date(mid_date, target_wy), "avg": monthly.loc[m]})
    out = pd.DataFrame(rows).dropna()
    out["plot_date"] = pd.to_datetime(out["plot_date"])
    return out.sort_values("plot_date")

# -- Widgets: Station 1 ---------------------------------------------------------
s1_header = widgets.HTML("<b>Station 1</b>")
s1_id     = widgets.Text(value="11464000", description="Station ID:",
                          style={"description_width": "90px"},
                          layout=widgets.Layout(width="240px"))
s1_color  = widgets.ColorPicker(value="#c76e4f", description="Color:",
                                 style={"description_width": "50px"},
                                 layout=widgets.Layout(width="160px"))
s1_style  = widgets.Dropdown(options=LINESTYLE_OPTIONS, value="-", description="Line style:",
                              style={"description_width": "70px"},
                              layout=widgets.Layout(width="160px"))
s1_label  = widgets.Text(value="", description="Label (opt):",
                          placeholder="Station 1 name",
                          style={"description_width": "90px"},
                          layout=widgets.Layout(width="300px"))

# -- Widgets: Station 2 (optional) ----------------------------------------------
s2_enable = widgets.Checkbox(value=True, description="Compare a second station",
                              style={"description_width": "initial"})
s2_box    = widgets.VBox([])

s2_id     = widgets.Text(value="11465350", description="Station ID:",
                          style={"description_width": "90px"},
                          layout=widgets.Layout(width="240px"))
s2_color  = widgets.ColorPicker(value="#5cb5bd", description="Color:",
                                 style={"description_width": "50px"},
                                 layout=widgets.Layout(width="160px"))
s2_style  = widgets.Dropdown(options=LINESTYLE_OPTIONS, value="-", description="Line style:",
                              style={"description_width": "70px"},
                              layout=widgets.Layout(width="160px"))
s2_label  = widgets.Text(value="", description="Label (opt):",
                          placeholder="Station 2 name",
                          style={"description_width": "90px"},
                          layout=widgets.Layout(width="300px"))

def _build_s2_box():
    return [widgets.HTML("<b>Station 2</b>"),
            widgets.HBox([s2_id, s2_color, s2_style, s2_label])]

s2_box.children = _build_s2_box()   # visible by default, matching s2_enable=True

def _toggle_s2(change):
    s2_box.children = _build_s2_box() if change["new"] else []
s2_enable.observe(_toggle_s2, names="value")

# -- Water year range (optional — hidden unless the checkbox is on) -------------
current_year = dt.date.today().year
use_custom_years = widgets.Checkbox(value=False,
                                     description="Specify water year range (uncheck to use all available data)",
                                     style={"description_width": "initial"})
wy_start_input = widgets.BoundedIntText(value=1991, min=1900, max=current_year + 1,
                                         description="History start (WY):",
                                         style={"description_width": "140px"},
                                         layout=widgets.Layout(width="260px"))
wy_end_input   = widgets.BoundedIntText(value=2021, min=1900, max=current_year + 1,
                                         description="End water year:",
                                         style={"description_width": "140px"},
                                         layout=widgets.Layout(width="260px"))
wy_box = widgets.VBox([])   # empty = hidden until the checkbox is checked

def _toggle_wy_inputs(change):
    wy_box.children = [widgets.HBox([wy_start_input, wy_end_input])] if change["new"] else []
use_custom_years.observe(_toggle_wy_inputs, names="value")

# -- Shared settings --------------------------------------------------------------
title_input = widgets.Text(value="", description="Title (opt):",
                            placeholder="e.g. Hydrologic Year Daily Flow",
                            style={"description_width": "90px"},
                            layout=widgets.Layout(width="500px"))
log_toggle  = widgets.Checkbox(value=True, description="Log scale (y-axis)",
                                style={"description_width": "initial"})

avg_line_toggle = widgets.Checkbox(value=False, description="Show monthly average line",
                                    style={"description_width": "initial"})
avg_line_color  = widgets.ColorPicker(value="#000000", description="Color:",
                                       style={"description_width": "50px"},
                                       layout=widgets.Layout(width="160px"))
avg_line_style  = widgets.Dropdown(options=LINESTYLE_OPTIONS, value="-", description="Line style:",
                                    style={"description_width": "70px"},
                                    layout=widgets.Layout(width="160px"))

run_button   = widgets.Button(description="Run", button_style="primary",
                               layout=widgets.Layout(width="90px"))
save_button  = widgets.Button(description="Save SVG", button_style="success",
                               layout=widgets.Layout(width="100px"))
status_label = widgets.Label(value="")
out          = widgets.Output()          # the plot renders here, under the cell
link_out     = widgets.Output()          # the download link renders here

ui = widgets.VBox([
    widgets.HTML(
        "<div style='display:flex;align-items:center;gap:14px;"
        "background:white;padding:10px 20px;border-radius:6px;"
        "border:2px solid #003B5C;margin-bottom:6px'>"
        f"<img src='data:image/png;base64,{_LOGO_B64}' "
        "style='height:42px;width:auto;display:block' alt='WRA logo'>"
        "<span style='color:#003B5C;font-size:16px;font-family:Verdana;font-weight:bold'>"
        "Historical Daily Flow Range Viewer</span></div>"
    ),
    widgets.HTML("<hr style='margin:4px 0'>"),
    s1_header,
    widgets.HBox([s1_id, s1_color, s1_style, s1_label]),
    s2_enable,
    s2_box,
    widgets.HTML("<hr style='margin:4px 0'>"),
    use_custom_years,
    wy_box,
    widgets.HBox([title_input, log_toggle]),
    widgets.HBox([avg_line_toggle, avg_line_color, avg_line_style]),
    widgets.HBox([run_button, save_button, status_label]),
    out,
    link_out,
])

_current_fig = None

# -- Core plot --------------------------------------------------------------------
def make_plot(site_data, sites_cfg, wy_start, wy_end, use_log, custom_title="",
              show_monthly_avg=False, monthly_avg_color="#000000",
              monthly_avg_style="-", monthly_avg_data=None):
    global _current_fig

    fig, ax = plt.subplots(figsize=(13, 6))

    line_handles, line_labels = [], []
    patch_handles, patch_labels = [], []

    for site_no, cfg in sites_cfg.items():
        stats = site_data[site_no]
        fill = ax.fill_between(stats["plot_date"], stats["min"], stats["max"],
                                color=cfg["band_color"], alpha=0.4)
        patch_handles.append(fill)
        patch_labels.append(f"{cfg['label']} - max and min")

        (line,) = ax.plot(stats["plot_date"], stats["mean"], color=cfg["mean_color"],
                           linestyle=cfg.get("line_style", "-"), linewidth=2)
        line_handles.append(line)
        line_labels.append(f"{cfg['label']} - daily averages")

        if show_monthly_avg and monthly_avg_data and site_no in monthly_avg_data:
            avg_series = monthly_avg_data[site_no]
            (avg_line,) = ax.plot(avg_series["plot_date"], avg_series["avg"],
                                   color=monthly_avg_color, linestyle=monthly_avg_style,
                                   linewidth=1.6, marker="o", markersize=3)
            line_handles.append(avg_line)
            line_labels.append(f"{cfg['label']} - monthly average")

    title_str = (custom_title.strip() if custom_title.strip() else
                 f"Hydrologic Year Daily Flow - Water Years {wy_start}-{wy_end}")
    ax.set_title(title_str)
    ax.set_xlabel("Month")
    ax.set_ylabel("Mean Daily Flow (cfs)")
    if use_log:
        ax.set_yscale("log")

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

    all_dates = pd.concat([site_data[s]["plot_date"] for s in sites_cfg])
    ax.set_xlim(all_dates.min(), all_dates.max())

    ax.grid(True, alpha=0.3)
    ax.legend(line_handles + patch_handles, line_labels + patch_labels,
              loc="upper right", fontsize=8, ncol=1)
    fig.tight_layout()

    fig.text(0.99, 0.01, f"© {dt.date.today().year} WRA, Inc.",
             ha="right", va="bottom", fontsize=7,
             fontfamily="Verdana", color="#888888")

    _current_fig = fig
    plt.show()

# -- Button callbacks ---------------------------------------------------------------
def on_run(b):
    with out:
        clear_output(wait=True)
        status_label.value = "Fetching data..."

        try:
            sites_cfg = {
                s1_id.value.strip(): {
                    "label": s1_label.value.strip() or f"Site {s1_id.value.strip()}",
                    "band_color": s1_color.value,
                    "mean_color": s1_color.value,
                    "line_style": s1_style.value,
                }
            }
            if s2_enable.value and s2_id.value.strip():
                sites_cfg[s2_id.value.strip()] = {
                    "label": s2_label.value.strip() or f"Site {s2_id.value.strip()}",
                    "band_color": s2_color.value,
                    "mean_color": s2_color.value,
                    "line_style": s2_style.value,
                }

            if not use_custom_years.value:
                history_start = "1900-01-01"
                end_date_str = None
            else:
                wy_start = wy_start_input.value
                wy_end = wy_end_input.value
                if wy_start > wy_end:
                    raise ValueError("History start (WY) must be <= End water year.")
                history_start = f"{wy_start - 1}-10-01"
                end_date_str = f"{wy_end}-09-30"

            raw_data = {}
            for site_no in sites_cfg:
                raw_data[site_no] = fetch_site_daily_flow(site_no, history_start, end_date_str)

            if not use_custom_years.value:
                # Derive the actual water-year span from whatever data came back,
                # and use the most recent complete year as the alignment reference.
                all_wy = pd.concat([raw["date"].apply(water_year_of) for raw in raw_data.values()])
                wy_start = int(all_wy.min())
                wy_end = int(all_wy.max())

            site_data = {}
            for site_no, raw in raw_data.items():
                site_data[site_no] = process_site(raw, wy_end)

            monthly_avg_data = None
            if avg_line_toggle.value:
                monthly_avg_data = {
                    site_no: build_monthly_avg_series(raw, wy_end)
                    for site_no, raw in raw_data.items()
                }

            make_plot(site_data, sites_cfg, wy_start, wy_end, log_toggle.value, title_input.value,
                      show_monthly_avg=avg_line_toggle.value,
                      monthly_avg_color=avg_line_color.value,
                      monthly_avg_style=avg_line_style.value,
                      monthly_avg_data=monthly_avg_data)

            for site_no, cfg in sites_cfg.items():
                raw = raw_data[site_no]
                total_start, total_end = fetch_period_of_record(site_no)
                print(f"\n{cfg['label']} (USGS {site_no})")
                display(build_summary_table(raw, total_start, total_end))

            ns = " | ".join(f"{cfg['label']}" for cfg in sites_cfg.values())
            status_label.value = f"Done - {ns} (WY {wy_start}-{wy_end})"

        except Exception as e:
            status_label.value = f"Error: {e}"
            raise

def on_save(b):
    global _current_fig
    if _current_fig is None:
        status_label.value = "Run the plot first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save hydrograph as SVG",
            defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            _current_fig.savefig(path, format="svg", bbox_inches="tight")
            status_label.value = f"Saved: {path}"
        else:
            status_label.value = "Save cancelled."
    except Exception as e:
        status_label.value = f"Save error: {e}"

run_button.on_click(on_run)
save_button.on_click(on_save)

# -- Launch -------------------------------------------------------------------------
display(ui)

In [5]:
# =============================================================================
#  WRA Streamflow Duration Hydrograph — Daily Percentiles (single station)
#  Rainbow/WRA/BrownBlue/Viridis percentile bands (3-7 bands, user's choice)
#  + an optional single-year daily-mean overlay, computed via the USGS NWIS
#  daily statistics service (no hyswap package needed).
# =============================================================================

import datetime as dt
import calendar

import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import rcParams

import ipywidgets as widgets
from IPython.display import display, clear_output

import tkinter as tk
from tkinter import filedialog

_LOGO_B64 = "..."

# -- WRA brand ------------------------------------------------------------------
WRA_NAVY = "#175536"
WRA_GOLD = "#C9A84C"

rcParams["font.family"] = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"] = 13
rcParams["axes.labelsize"] = 11
rcParams["xtick.labelsize"] = 8
rcParams["ytick.labelsize"] = 8

# -- Color palettes, low flow -> high flow (7-point gradient; fewer bands are
#    sampled evenly from this same gradient, see dur_get_palette_colors) -----
DUR_COLOR_PALETTES = {
    "rainbow":   ["#D7191C", "#FD8D3C", "#FED976", "#78C679", "#41B6C4", "#2C7FB8", "#6A51A3"],
    "wra":       ["#7D3B2E", "#C76E4F", "#D4822B", "#A19100", "#529E2E", "#5CB5BD", "#0A596A"],
    "brownblue": ["#8C510A", "#D8B365", "#F6E8C3", "#F5F5F5", "#C7EAE5", "#5AB4AC", "#01665E"],
    "viridis":   ["#440154", "#443983", "#31688E", "#21918C", "#35B779", "#90D743", "#FDE725"],
}
DUR_PALETTE_LABELS = [
    ("Rainbow", "rainbow"),
    ("WRA Colors", "wra"),
    ("BrownBlue (hyswap default)", "brownblue"),
    ("Viridis (colorblind-friendly)", "viridis"),
]

# -- Percentile-band boundary sets, by number of bands requested -------------
# p25/p75 are always present, so the "normal" band is always identifiable.
DUR_BAND_BOUNDARIES = {
    3: ["min_va", "p25_va", "p75_va", "max_va"],
    4: ["min_va", "p10_va", "p25_va", "p75_va", "max_va"],
    5: ["min_va", "p10_va", "p25_va", "p75_va", "p90_va", "max_va"],
    6: ["min_va", "p05_va", "p10_va", "p25_va", "p75_va", "p90_va", "max_va"],
    7: ["min_va", "p05_va", "p10_va", "p25_va", "p75_va", "p90_va", "p95_va", "max_va"],
}


def dur_get_palette_colors(name, n_bands):
    """Sample n_bands colors evenly from the 7-color gradient so any band
    count (3-7) still reads as a smooth low-flow -> high-flow gradient."""
    full = DUR_COLOR_PALETTES.get(name, DUR_COLOR_PALETTES["rainbow"])
    idx = np.round(np.linspace(0, len(full) - 1, n_bands)).astype(int)
    return [full[i] for i in idx]


def dur_band_defs(n_bands):
    """Build the (lower, upper, label) tuples for the requested band count."""
    bounds = DUR_BAND_BOUNDARIES[n_bands]

    def pretty(col):
        return {"min_va": "Min", "max_va": "Max"}.get(col, col.replace("_va", "").upper())

    defs = []
    for lower, upper in zip(bounds[:-1], bounds[1:]):
        label = f"{pretty(lower)} – {pretty(upper)}"
        if lower == "p25_va" and upper == "p75_va":
            label += "  (normal)"
        defs.append((lower, upper, label))
    return defs


DUR_LINESTYLE_OPTIONS = [("Solid", "-"), ("Dashed", "--"), ("Dash-dot", "-."), ("Dotted", ":")]


def dur_build_monthly_table(pctl_df):
    """Monthly average/maximum/minimum flow, built from the day-of-year
    percentile table (average = mean of each day's mean_va; max/min = the
    highest max_va / lowest min_va seen among that month's days)."""
    g = pctl_df.groupby("month_nu").agg(
        **{"Average (cfs)": ("mean_va", "mean"),
           "Maximum (cfs)": ("max_va", "max"),
           "Minimum (cfs)": ("min_va", "min")}
    )
    g = g.reindex(range(1, 13))
    g.index = [calendar.month_abbr[m] for m in g.index]
    g.index.name = "Month"
    return g.round(1)


def dur_fetch_period_of_record(site_no, parameter_cd="00060"):
    """Full available period of record for a site/parameter, from the NWIS
    site service's series catalog (a lightweight metadata call)."""
    url = "https://waterservices.usgs.gov/nwis/site/"
    params = {"sites": site_no, "format": "rdb",
              "seriesCatalogOutput": "true", "parameterCd": parameter_cd}
    try:
        r = requests.get(url, params=params, timeout=15)
        lines = [l for l in r.text.splitlines() if not l.startswith("#") and l.strip()]
        if len(lines) < 3:
            return None, None
        header = lines[0].split("\t")
        rows = [l.split("\t") for l in lines[2:] if l.strip()]
        if not rows:
            return None, None
        df = pd.DataFrame(rows, columns=header[: len(rows[0])])
        if "begin_date" not in df.columns or "end_date" not in df.columns:
            return None, None
        begin = pd.to_datetime(df["begin_date"], errors="coerce").min()
        end = pd.to_datetime(df["end_date"], errors="coerce").max()
        if pd.isna(begin) or pd.isna(end):
            return None, None
        return dur_water_year_of(begin.date()), dur_water_year_of(end.date())
    except Exception:
        return None, None


def dur_build_summary_table(pctl_df, total_start_wy, total_end_wy):
    """The monthly table with two extra rows on top: the full period of
    record available for the site, and the period actually used here."""
    monthly = dur_build_monthly_table(pctl_df)
    used_start = int(pctl_df["begin_yr"].min())
    used_end = int(pctl_df["end_yr"].max())
    total_txt = (f"WY{total_start_wy}-WY{total_end_wy}" if total_start_wy else "Unknown")
    info = pd.DataFrame(
        {"Average (cfs)": [total_txt, f"WY{used_start}-WY{used_end}"],
         "Maximum (cfs)": ["—", "—"],
         "Minimum (cfs)": ["—", "—"]},
        index=["Total period available", "Period used"],
    )
    info.index.name = "Month"
    return pd.concat([info, monthly])


def dur_build_monthly_avg_series(pctl_df, plot_year, year_type):
    """Monthly-average flow, one point per month (mid-month), placed on the
    same plot-date axis as the rest of the chart."""
    monthly = pctl_df.groupby("month_nu")["mean_va"].mean().reindex(range(1, 13))
    rows = []
    for m in range(1, 13):
        rows.append({
            "plot_date": dur_plot_date(m, 15, plot_year, year_type),
            "avg": monthly.loc[m],
        })
    out = pd.DataFrame(rows).dropna()
    out["plot_date"] = pd.to_datetime(out["plot_date"])
    return out.sort_values("plot_date")


def dur_palette_swatch_html(name, n_bands):
    colors = dur_get_palette_colors(name, n_bands)
    swatches = "".join(
        f"<div style='width:22px;height:20px;background:{c};"
        "display:inline-block;border:1px solid #999;margin-right:1px'></div>"
        for c in colors
    )
    return f"<div style='margin-top:3px'>{swatches}</div>"


# -- Helpers ----------------------------------------------------------------------
def dur_water_year_of(date):
    return date.year + 1 if date.month >= 10 else date.year


def dur_plot_date(month, day, ref_year, year_type):
    """Map a (month, day) onto a single reference year's calendar so an
    entire water-year or calendar-year cycle can share one x-axis."""
    month, day, ref_year = int(month), int(day), int(ref_year)
    yr = ref_year - 1 if (year_type == "water" and month >= 10) else ref_year
    try:
        return dt.date(yr, month, day)
    except ValueError:      # Feb 29 landing on a non-leap reference year
        return dt.date(yr, 2, 28)


def dur_fetch_daily_percentiles(site_no, start_wy=None, end_wy=None):
    """
    Pull day-of-year percentile statistics (min, p05, p10, p25, p50, p75,
    p90, p95, max) straight from the USGS NWIS daily statistics service.
    If start_wy/end_wy are given, they restrict which calendar dates feed
    into each day-of-year bucket (statYearType isn't valid for daily
    reports, so this is a calendar-year approximation of a water-year
    range, off by at most a few months at each end).
    """
    url = "https://waterservices.usgs.gov/nwis/stat/"
    params = {
        "format": "rdb",
        "sites": site_no,
        "statReportType": "daily",
        "statTypeCd": "all",
        "parameterCd": "00060",
    }
    if start_wy is not None and end_wy is not None:
        params["startDt"] = str(int(start_wy))
        params["endDt"] = str(int(end_wy))

    r = requests.get(url, params=params, timeout=30)
    if not r.ok:
        raise ValueError(
            f"USGS statistics service returned {r.status_code} for site {site_no}: "
            f"{r.text.strip()[:300]}"
        )

    lines = [l for l in r.text.splitlines() if not l.startswith("#") and l.strip()]
    if len(lines) < 3:
        raise ValueError(f"No daily statistics returned for site {site_no}.")

    header = lines[0].split("\t")
    data_lines = [l for l in lines[2:] if l.strip()]
    rows = [l.split("\t") for l in data_lines]
    df = pd.DataFrame(rows, columns=header[: len(rows[0])])

    needed = ["month_nu", "day_nu", "begin_yr", "end_yr",
              "min_va", "p05_va", "p10_va", "p25_va", "p50_va",
              "p75_va", "p90_va", "p95_va", "max_va", "mean_va"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Statistics service is missing expected columns: {missing}")

    for c in needed:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["month_nu", "day_nu"]).copy()
    df["month_nu"] = df["month_nu"].astype(int)
    df["day_nu"] = df["day_nu"].astype(int)
    return df[needed]


def dur_fetch_daily_flow(site_no, start, end=None):
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "json",
        "sites": site_no,
        "startDT": start,
        "parameterCd": "00060",
        "statCd": "00003",
    }
    if end:
        params["endDT"] = end
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    values = data["value"]["timeSeries"][0]["values"][0]["value"]
    df = pd.DataFrame(values)
    if df.empty:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
    df["flow_cfs"] = pd.to_numeric(df["value"])
    return df.rename(columns={"dateTime": "date"})[["date", "flow_cfs"]]


def dur_fetch_station_name(site_no):
    url = "https://waterservices.usgs.gov/nwis/site/"
    params = {"sites": site_no, "format": "rdb", "siteOutput": "expanded"}
    try:
        r = requests.get(url, params=params, timeout=15)
        for line in r.text.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) > 2 and parts[0] == site_no:
                return parts[2].strip().title()
    except Exception:
        pass
    return f"USGS Station {site_no}"


# -- Widgets: single station ------------------------------------------------------
dur_site_id = widgets.Text(value="11169500", description="Station ID:",
                            style={"description_width": "90px"},
                            layout=widgets.Layout(width="240px"))
dur_label = widgets.Text(value="", description="Label (opt):",
                          placeholder="e.g. San Francisquito Ck at Stanford",
                          style={"description_width": "90px"},
                          layout=widgets.Layout(width="340px"))

dur_year_type = widgets.Dropdown(
    options=[("Water Year (Oct–Sep)", "water"), ("Calendar Year (Jan–Dec)", "calendar")],
    value="water", description="Year type:",
    style={"description_width": "90px"}, layout=widgets.Layout(width="260px"),
)

_dur_today = dt.date.today()

# -- Optional "year to plot" overlay --------------------------------------------
dur_show_year_toggle = widgets.Checkbox(
    value=False, description="Overlay a specific year's daily flow",
    style={"description_width": "initial"},
)
dur_plot_year = widgets.BoundedIntText(
    value=dur_water_year_of(_dur_today), min=1900, max=_dur_today.year + 1,
    description="Year to plot:", style={"description_width": "90px"},
    layout=widgets.Layout(width="220px"),
)
dur_year_color = widgets.ColorPicker(value="#000000", description="Color:",
                                      style={"description_width": "50px"},
                                      layout=widgets.Layout(width="160px"))
dur_year_style = widgets.Dropdown(options=DUR_LINESTYLE_OPTIONS, value="-",
                                   description="Line style:",
                                   style={"description_width": "70px"},
                                   layout=widgets.Layout(width="160px"))
dur_year_box = widgets.VBox([])  # hidden by default, matching dur_show_year_toggle=False


def _dur_toggle_year_box(change):
    dur_year_box.children = (
        [widgets.HBox([dur_plot_year, dur_year_color, dur_year_style])] if change["new"] else []
    )


dur_show_year_toggle.observe(_dur_toggle_year_box, names="value")

# -- Number of percentile bands ---------------------------------------------------
dur_nbands_choice = widgets.Dropdown(
    options=[3, 4, 5, 6, 7], value=7, description="# of bands:",
    style={"description_width": "90px"}, layout=widgets.Layout(width="160px"),
)

dur_palette_choice = widgets.Dropdown(
    options=DUR_PALETTE_LABELS,
    value="rainbow", description="Color palette:",
    style={"description_width": "95px"}, layout=widgets.Layout(width="230px"),
)
dur_palette_swatch = widgets.HTML(
    value=dur_palette_swatch_html(dur_palette_choice.value, dur_nbands_choice.value)
)


def _dur_on_palette_or_bands_change(change):
    dur_palette_swatch.value = dur_palette_swatch_html(
        dur_palette_choice.value, dur_nbands_choice.value
    )


dur_palette_choice.observe(_dur_on_palette_or_bands_change, names="value")
dur_nbands_choice.observe(_dur_on_palette_or_bands_change, names="value")

dur_log_toggle = widgets.Checkbox(value=True, description="Log scale (y-axis)",
                                   style={"description_width": "initial"})
dur_median_toggle = widgets.Checkbox(value=True, description="Show median (P50) line",
                                      style={"description_width": "initial"})
dur_median_color = widgets.ColorPicker(value="#FFFFFF", description="Color:",
                                        style={"description_width": "50px"},
                                        layout=widgets.Layout(width="160px"))
dur_median_style = widgets.Dropdown(options=DUR_LINESTYLE_OPTIONS, value="--",
                                     description="Line style:",
                                     style={"description_width": "70px"},
                                     layout=widgets.Layout(width="160px"))

dur_monthly_avg_toggle = widgets.Checkbox(value=False, description="Show monthly average line",
                                           style={"description_width": "initial"})
dur_monthly_avg_color = widgets.ColorPicker(value="#000000", description="Color:",
                                             style={"description_width": "50px"},
                                             layout=widgets.Layout(width="160px"))
dur_monthly_avg_style = widgets.Dropdown(options=DUR_LINESTYLE_OPTIONS, value="-.",
                                          description="Line style:",
                                          style={"description_width": "70px"},
                                          layout=widgets.Layout(width="160px"))

# -- Optional historical water-year range (hidden unless checked) ---------------
dur_custom_years_toggle = widgets.Checkbox(
    value=False,
    description="Specify years for historical percentiles (default: full period of record)",
    style={"description_width": "initial"},
)
dur_hist_start_wy = widgets.BoundedIntText(
    value=1990, min=1900, max=_dur_today.year + 1, description="History start:",
    style={"description_width": "100px"}, layout=widgets.Layout(width="220px"),
)
dur_hist_end_wy = widgets.BoundedIntText(
    value=_dur_today.year, min=1900, max=_dur_today.year + 1, description="History end:",
    style={"description_width": "100px"}, layout=widgets.Layout(width="220px"),
)
dur_hist_range_box = widgets.VBox([])


def _dur_toggle_hist_range(change):
    dur_hist_range_box.children = (
        [widgets.HBox([dur_hist_start_wy, dur_hist_end_wy])] if change["new"] else []
    )


dur_custom_years_toggle.observe(_dur_toggle_hist_range, names="value")

dur_title_input = widgets.Text(value="", description="Title (opt):",
                                placeholder="e.g. Streamflow Percentiles - Site 11169500",
                                style={"description_width": "90px"},
                                layout=widgets.Layout(width="500px"))

dur_run_button = widgets.Button(description="Run", button_style="primary",
                                 layout=widgets.Layout(width="90px"))
dur_save_button = widgets.Button(description="Save SVG", button_style="success",
                                  layout=widgets.Layout(width="100px"))
dur_status = widgets.Label(value="")
dur_out = widgets.Output()

dur_ui = widgets.VBox([
    widgets.HTML(
        "<div style='display:flex;align-items:center;gap:14px;"
        "background:white;padding:10px 20px;border-radius:6px;"
        f"border:2px solid {WRA_NAVY};margin-bottom:6px'>"
        f"<img src='data:image/png;base64,{_LOGO_B64}' "
        "style='height:42px;width:auto;display:block' alt='WRA logo'>"
        f"<span style='color:{WRA_NAVY};font-size:16px;"
        "font-family:Verdana;font-weight:bold'>"
        "Streamflow Duration Hydrograph — Daily Percentiles</span></div>"
    ),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([dur_site_id, dur_label]),
    widgets.HBox([dur_year_type]),
    dur_show_year_toggle,
    dur_year_box,
    widgets.HBox([dur_nbands_choice, dur_palette_choice, dur_palette_swatch]),
    widgets.HBox([dur_log_toggle]),
    widgets.HBox([dur_median_toggle, dur_median_color, dur_median_style]),
    widgets.HBox([dur_monthly_avg_toggle, dur_monthly_avg_color, dur_monthly_avg_style]),
    dur_custom_years_toggle,
    dur_hist_range_box,
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([dur_title_input]),
    widgets.HBox([dur_run_button, dur_save_button, dur_status]),
    dur_out,
])

_dur_current_fig = None


# -- Core plot ----------------------------------------------------------------------
def dur_make_plot(pctl_aligned, flow_aligned, site_label, year_type, plot_year,
                   use_log, palette_name, show_median, n_bands, custom_title="",
                   median_color="#FFFFFF", median_style="--",
                   year_color="#000000", year_style="-",
                   show_monthly_avg=False, monthly_avg_color="#000000",
                   monthly_avg_style="-.", monthly_avg_series=None):
    global _dur_current_fig

    colors = dur_get_palette_colors(palette_name, n_bands)
    band_defs = dur_band_defs(n_bands)

    fig, ax = plt.subplots(figsize=(13, 6))

    for (lower, upper, _label), color in zip(band_defs, colors):
        ax.fill_between(pctl_aligned["plot_date"], pctl_aligned[lower],
                         pctl_aligned[upper], color=color, alpha=0.85,
                         linewidth=0, zorder=1)

    if show_median:
        ax.plot(pctl_aligned["plot_date"], pctl_aligned["p50_va"],
                color=median_color, linewidth=1.0, linestyle=median_style, alpha=0.8,
                zorder=2, label="Median (P50)")

    if show_monthly_avg and monthly_avg_series is not None:
        ax.plot(monthly_avg_series["plot_date"], monthly_avg_series["avg"],
                color=monthly_avg_color, linewidth=1.6, linestyle=monthly_avg_style,
                marker="o", markersize=3, zorder=4, label="Monthly average")

    if flow_aligned is not None:
        yr_label = f"Water Year {plot_year}" if year_type == "water" else f"{plot_year}"
        ax.plot(flow_aligned["plot_date"], flow_aligned["flow_cfs"],
                color=year_color, linewidth=1.8, linestyle=year_style,
                zorder=5, label=f"{yr_label} daily mean")

    if use_log:
        ax.set_yscale("log")

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.set_xlim(pctl_aligned["plot_date"].min(), pctl_aligned["plot_date"].max())
    ax.set_xlabel("Month")
    ax.set_ylabel("Discharge (cfs)")

    title_str = (custom_title.strip() if custom_title.strip() else
                 f"Streamflow Percentiles by Day of Year - {site_label}")
    ax.set_title(title_str)
    ax.grid(True, which="both", alpha=0.25)

    band_handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.85) for c in colors]
    band_labels = [lbl for _, _, lbl in band_defs]
    line_handles, line_labels = ax.get_legend_handles_labels()
    ax.legend(list(reversed(band_handles)) + line_handles,
              list(reversed(band_labels)) + line_labels,
              loc="upper right", fontsize=7, framealpha=0.9, ncol=1)

    fig.text(0.99, 0.01, f"© {dt.date.today().year} WRA, Inc.",
              ha="right", va="bottom", fontsize=7,
              fontfamily="Verdana", color="#888888")

    fig.tight_layout()
    _dur_current_fig = fig
    plt.show()


# -- Button callbacks -----------------------------------------------------------------
def dur_on_run(b):
    with dur_out:
        clear_output(wait=True)
        dur_status.value = "Fetching historical percentiles..."
        try:
            site_no = dur_site_id.value.strip()
            if not site_no:
                dur_status.value = "Please enter a Station ID."
                return

            year_type = dur_year_type.value
            show_year = dur_show_year_toggle.value
            n_bands = int(dur_nbands_choice.value)

            if show_year:
                plot_year = int(dur_plot_year.value)
            else:
                # No overlay year requested -- still need a reference year to
                # lay out the x-axis (Oct-Sep vs Jan-Dec), just not shown.
                plot_year = (dur_water_year_of(_dur_today)
                             if year_type == "water" else _dur_today.year)

            if dur_custom_years_toggle.value:
                hist_start_wy = int(dur_hist_start_wy.value)
                hist_end_wy = int(dur_hist_end_wy.value)
                if hist_start_wy > hist_end_wy:
                    raise ValueError("History start must be <= History end.")
            else:
                hist_start_wy = None
                hist_end_wy = None

            pctl = dur_fetch_daily_percentiles(site_no, hist_start_wy, hist_end_wy)
            pctl["plot_date"] = pctl.apply(
                lambda r: dur_plot_date(r["month_nu"], r["day_nu"], plot_year, year_type),
                axis=1,
            )
            pctl["plot_date"] = pd.to_datetime(pctl["plot_date"])
            pctl = pctl.sort_values("plot_date")

            flow = None
            if show_year:
                dur_status.value = "Fetching daily mean flow for selected year..."
                if year_type == "water":
                    start = f"{plot_year - 1}-10-01"
                    end = f"{plot_year}-09-30"
                else:
                    start = f"{plot_year}-01-01"
                    end = f"{plot_year}-12-31"

                flow = dur_fetch_daily_flow(site_no, start, end)
                flow["plot_date"] = flow["date"].apply(
                    lambda d: dur_plot_date(d.month, d.day, plot_year, year_type)
                )
                flow["plot_date"] = pd.to_datetime(flow["plot_date"])
                flow = flow.sort_values("plot_date")

            site_label = dur_label.value.strip() or dur_fetch_station_name(site_no)

            monthly_avg_series = None
            if dur_monthly_avg_toggle.value:
                monthly_avg_series = dur_build_monthly_avg_series(pctl, plot_year, year_type)

            dur_status.value = "Rendering chart..."
            dur_make_plot(pctl, flow, site_label, year_type, plot_year,
                          dur_log_toggle.value, dur_palette_choice.value,
                          dur_median_toggle.value, n_bands, dur_title_input.value,
                          median_color=dur_median_color.value, median_style=dur_median_style.value,
                          year_color=dur_year_color.value, year_style=dur_year_style.value,
                          show_monthly_avg=dur_monthly_avg_toggle.value,
                          monthly_avg_color=dur_monthly_avg_color.value,
                          monthly_avg_style=dur_monthly_avg_style.value,
                          monthly_avg_series=monthly_avg_series)

            span = f"WY{pctl['begin_yr'].min():.0f}-WY{pctl['end_yr'].max():.0f}"
            if dur_custom_years_toggle.value:
                total_start_wy, total_end_wy = dur_fetch_period_of_record(site_no)
            else:
                total_start_wy = int(pctl["begin_yr"].min())
                total_end_wy = int(pctl["end_yr"].max())
            print(f"\n{site_label} (USGS {site_no})")
            display(dur_build_summary_table(pctl, total_start_wy, total_end_wy))

            dur_status.value = f"Done - {site_label} (historical record: {span})"

        except Exception as e:
            dur_status.value = f"Error: {e}"
            raise


def dur_on_save(b):
    global _dur_current_fig
    if _dur_current_fig is None:
        dur_status.value = "Run the plot first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save duration hydrograph as SVG",
            defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            _dur_current_fig.savefig(path, format="svg", bbox_inches="tight")
            dur_status.value = f"Saved: {path}"
        else:
            dur_status.value = "Save cancelled."
    except Exception as e:
        dur_status.value = f"Save error: {e}"


dur_run_button.on_click(dur_on_run)
dur_save_button.on_click(dur_on_save)

# -- Launch -------------------------------------------------------------------------
display(dur_ui)

In [6]:
# =============================================================================
#  WRA Streamflow Duration Hydrograph — Daily Percentiles (TWO stations, COMBINED)
#  Adds the daily flow of two stations together (e.g. two tributaries above a
#  confluence) and computes percentile bands (3-7, user's choice) and an
#  optional current-year trace from that COMBINED daily series.
#
#  NWIS's stat service only computes percentiles for real gauges, so this
#  pulls full daily-value history for both stations, sums them on matching
#  dates, and computes the percentiles locally with pandas.
# =============================================================================

import datetime as dt
import calendar

import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import rcParams

import ipywidgets as widgets
from IPython.display import display, clear_output

import tkinter as tk
from tkinter import filedialog

_LOGO_B64 = "..."

# -- WRA brand ------------------------------------------------------------------
WRA_NAVY = "#175536"
WRA_GOLD = "#C9A84C"

rcParams["font.family"] = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"] = 13
rcParams["axes.labelsize"] = 11
rcParams["xtick.labelsize"] = 8
rcParams["ytick.labelsize"] = 8

# -- Color palettes, low flow -> high flow (7-point gradient; fewer bands are
#    sampled evenly from this same gradient, see dur2_get_palette_colors) ---
DUR2_COLOR_PALETTES = {
    "rainbow":   ["#D7191C", "#FD8D3C", "#FED976", "#78C679", "#41B6C4", "#2C7FB8", "#6A51A3"],
    "wra":       ["#7D3B2E", "#C76E4F", "#D4822B", "#A19100", "#529E2E", "#5CB5BD", "#0A596A"],
    "brownblue": ["#8C510A", "#D8B365", "#F6E8C3", "#F5F5F5", "#C7EAE5", "#5AB4AC", "#01665E"],
    "viridis":   ["#440154", "#443983", "#31688E", "#21918C", "#35B779", "#90D743", "#FDE725"],
}
DUR2_PALETTE_LABELS = [
    ("Rainbow", "rainbow"),
    ("WRA Colors", "wra"),
    ("BrownBlue (hyswap default)", "brownblue"),
    ("Viridis (colorblind-friendly)", "viridis"),
]

# -- Percentile-band boundary sets, by number of bands requested -------------
# p25/p75 are always present, so the "normal" band is always identifiable.
DUR2_BAND_BOUNDARIES = {
    3: ["min_va", "p25_va", "p75_va", "max_va"],
    4: ["min_va", "p10_va", "p25_va", "p75_va", "max_va"],
    5: ["min_va", "p10_va", "p25_va", "p75_va", "p90_va", "max_va"],
    6: ["min_va", "p05_va", "p10_va", "p25_va", "p75_va", "p90_va", "max_va"],
    7: ["min_va", "p05_va", "p10_va", "p25_va", "p75_va", "p90_va", "p95_va", "max_va"],
}


def dur2_get_palette_colors(name, n_bands):
    """Sample n_bands colors evenly from the 7-color gradient so any band
    count (3-7) still reads as a smooth low-flow -> high-flow gradient."""
    full = DUR2_COLOR_PALETTES.get(name, DUR2_COLOR_PALETTES["rainbow"])
    idx = np.round(np.linspace(0, len(full) - 1, n_bands)).astype(int)
    return [full[i] for i in idx]


def dur2_band_defs(n_bands):
    """Build the (lower, upper, label) tuples for the requested band count."""
    bounds = DUR2_BAND_BOUNDARIES[n_bands]

    def pretty(col):
        return {"min_va": "Min", "max_va": "Max"}.get(col, col.replace("_va", "").upper())

    defs = []
    for lower, upper in zip(bounds[:-1], bounds[1:]):
        label = f"{pretty(lower)} – {pretty(upper)}"
        if lower == "p25_va" and upper == "p75_va":
            label += "  (normal)"
        defs.append((lower, upper, label))
    return defs


DUR2_LINESTYLE_OPTIONS = [("Solid", "-"), ("Dashed", "--"), ("Dash-dot", "-."), ("Dotted", ":")]


def dur2_build_monthly_table(combined_df):
    """Monthly average/maximum/minimum COMBINED daily flow, computed across
    every year in the (possibly range-restricted) combined series."""
    d = combined_df.copy()
    d["month"] = d["date"].dt.month
    g = d.groupby("month")["flow_cfs"].agg(["mean", "max", "min"])
    g = g.reindex(range(1, 13))
    g.index = [calendar.month_abbr[m] for m in g.index]
    g.index.name = "Month"
    g.columns = ["Average (cfs)", "Maximum (cfs)", "Minimum (cfs)"]
    return g.round(1)


def dur2_fetch_period_of_record(site_no, parameter_cd="00060"):
    """Full available period of record for a site/parameter, from the NWIS
    site service's series catalog (a lightweight metadata call)."""
    url = "https://waterservices.usgs.gov/nwis/site/"
    params = {"sites": site_no, "format": "rdb",
              "seriesCatalogOutput": "true", "parameterCd": parameter_cd}
    try:
        r = requests.get(url, params=params, timeout=15)
        lines = [l for l in r.text.splitlines() if not l.startswith("#") and l.strip()]
        if len(lines) < 3:
            return None, None
        header = lines[0].split("\t")
        rows = [l.split("\t") for l in lines[2:] if l.strip()]
        if not rows:
            return None, None
        df = pd.DataFrame(rows, columns=header[: len(rows[0])])
        if "begin_date" not in df.columns or "end_date" not in df.columns:
            return None, None
        begin = pd.to_datetime(df["begin_date"], errors="coerce").min()
        end = pd.to_datetime(df["end_date"], errors="coerce").max()
        if pd.isna(begin) or pd.isna(end):
            return None, None
        return dur2_water_year_of(begin.date()), dur2_water_year_of(end.date())
    except Exception:
        return None, None


def dur2_build_summary_table(table_data, site1, site2, total1, total2,
                              rec_start_wy, rec_end_wy):
    """The monthly table with three extra rows on top: each station's own
    full period of record, and the combined water-year range actually used."""
    monthly = dur2_build_monthly_table(table_data)
    t1_txt = f"WY{total1[0]}-WY{total1[1]}" if total1[0] else "Unknown"
    t2_txt = f"WY{total2[0]}-WY{total2[1]}" if total2[0] else "Unknown"
    info = pd.DataFrame(
        {"Average (cfs)": [f"Station {site1}: {t1_txt}",
                            f"Station {site2}: {t2_txt}",
                            f"WY{rec_start_wy}-WY{rec_end_wy}"],
         "Maximum (cfs)": ["—", "—", "—"],
         "Minimum (cfs)": ["—", "—", "—"]},
        index=["Station 1 total period available",
               "Station 2 total period available",
               "Combined period used"],
    )
    info.index.name = "Month"
    return pd.concat([info, monthly])


def dur2_build_monthly_avg_series(combined_df, plot_year, year_type):
    """Monthly-average COMBINED flow, one point per month (mid-month),
    placed on the same plot-date axis as the rest of the chart."""
    d = combined_df.copy()
    d["month"] = d["date"].dt.month
    monthly = d.groupby("month")["flow_cfs"].mean().reindex(range(1, 13))
    rows = []
    for m in range(1, 13):
        rows.append({
            "plot_date": dur2_plot_date(m, 15, plot_year, year_type),
            "avg": monthly.loc[m],
        })
    out = pd.DataFrame(rows).dropna()
    out["plot_date"] = pd.to_datetime(out["plot_date"])
    return out.sort_values("plot_date")


def dur2_palette_swatch_html(name, n_bands):
    colors = dur2_get_palette_colors(name, n_bands)
    swatches = "".join(
        f"<div style='width:22px;height:20px;background:{c};"
        "display:inline-block;border:1px solid #999;margin-right:1px'></div>"
        for c in colors
    )
    return f"<div style='margin-top:3px'>{swatches}</div>"


# -- Helpers ----------------------------------------------------------------------
def dur2_water_year_of(date):
    return date.year + 1 if date.month >= 10 else date.year


def dur2_plot_date(month, day, ref_year, year_type):
    """Map a (month, day) onto a single reference year's calendar."""
    month, day, ref_year = int(month), int(day), int(ref_year)
    yr = ref_year - 1 if (year_type == "water" and month >= 10) else ref_year
    try:
        return dt.date(yr, month, day)
    except ValueError:      # Feb 29 landing on a non-leap reference year
        return dt.date(yr, 2, 28)


def dur2_fetch_full_daily_flow(site_no, start_date="1890-01-01", end_date=None):
    """Pull the FULL available daily-mean-flow history for one station
    (as opposed to a single year) so it can be summed with another
    station's history before computing percentiles."""
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "json",
        "sites": site_no,
        "startDT": start_date,
        "parameterCd": "00060",
        "statCd": "00003",
    }
    if end_date:
        params["endDT"] = end_date
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    ts = data.get("value", {}).get("timeSeries", [])
    if not ts:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    values = ts[0]["values"][0]["value"]
    df = pd.DataFrame(values)
    if df.empty:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
    df["flow_cfs"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["flow_cfs"])
    return df.rename(columns={"dateTime": "date"})[["date", "flow_cfs"]]


def dur2_combine_two_stations(df1, df2):
    """Sum two stations' daily flow on matching dates only (inner join) --
    a date only counts if BOTH stations reported a value that day."""
    merged = pd.merge(df1, df2, on="date", suffixes=("_1", "_2"))
    if merged.empty:
        raise ValueError(
            "The two stations have no overlapping dates with data -- "
            "cannot combine them."
        )
    merged["flow_cfs"] = merged["flow_cfs_1"] + merged["flow_cfs_2"]
    return merged[["date", "flow_cfs"]].sort_values("date")


def dur2_compute_percentiles(combined_df, start_wy=None, end_wy=None):
    """Compute day-of-year percentile statistics locally from a combined
    daily flow series (mirrors what the NWIS stat service returns for a
    single real gauge, but for our virtual combined 'gauge')."""
    df = combined_df.copy()
    df["water_year"] = df["date"].apply(lambda d: dur2_water_year_of(d))
    if start_wy is not None and end_wy is not None:
        df = df[(df["water_year"] >= start_wy) & (df["water_year"] <= end_wy)]
    if df.empty:
        raise ValueError("No combined data available in the requested water-year range.")

    df["month_nu"] = df["date"].dt.month
    df["day_nu"] = df["date"].dt.day

    records = []
    for (m, d), grp in df.groupby(["month_nu", "day_nu"]):
        s = grp["flow_cfs"].dropna()
        if s.empty:
            continue
        records.append({
            "month_nu": m, "day_nu": d,
            "min_va": s.min(),
            "p05_va": s.quantile(0.05),
            "p10_va": s.quantile(0.10),
            "p25_va": s.quantile(0.25),
            "p50_va": s.quantile(0.50),
            "p75_va": s.quantile(0.75),
            "p90_va": s.quantile(0.90),
            "p95_va": s.quantile(0.95),
            "max_va": s.max(),
            "n_years": s.shape[0],
        })
    out = pd.DataFrame(records)
    if out.empty:
        raise ValueError("Could not compute percentiles from the combined data.")
    return out, int(df["water_year"].min()), int(df["water_year"].max())


def dur2_fetch_station_name(site_no):
    url = "https://waterservices.usgs.gov/nwis/site/"
    params = {"sites": site_no, "format": "rdb", "siteOutput": "expanded"}
    try:
        r = requests.get(url, params=params, timeout=15)
        for line in r.text.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) > 2 and parts[0] == site_no:
                return parts[2].strip().title()
    except Exception:
        pass
    return f"USGS Station {site_no}"


# -- Widgets: two stations, combined ------------------------------------------------
dur2_site1_id = widgets.Text(value="", description="Station 1 ID:",
                              placeholder="e.g. 11169500",
                              style={"description_width": "100px"},
                              layout=widgets.Layout(width="250px"))
dur2_site2_id = widgets.Text(value="", description="Station 2 ID:",
                              placeholder="e.g. 11169800",
                              style={"description_width": "100px"},
                              layout=widgets.Layout(width="250px"))
dur2_label = widgets.Text(value="", description="Label (opt):",
                           placeholder="e.g. Combined flow above confluence",
                           style={"description_width": "90px"},
                           layout=widgets.Layout(width="380px"))

dur2_year_type = widgets.Dropdown(
    options=[("Water Year (Oct–Sep)", "water"), ("Calendar Year (Jan–Dec)", "calendar")],
    value="water", description="Year type:",
    style={"description_width": "90px"}, layout=widgets.Layout(width="260px"),
)

_dur2_today = dt.date.today()

# -- Optional "year to plot" overlay --------------------------------------------
dur2_show_year_toggle = widgets.Checkbox(
    value=False, description="Overlay a specific year's combined daily flow",
    style={"description_width": "initial"},
)
dur2_plot_year = widgets.BoundedIntText(
    value=dur2_water_year_of(_dur2_today), min=1900, max=_dur2_today.year + 1,
    description="Year to plot:", style={"description_width": "90px"},
    layout=widgets.Layout(width="220px"),
)
dur2_year_color = widgets.ColorPicker(value="#000000", description="Color:",
                                       style={"description_width": "50px"},
                                       layout=widgets.Layout(width="160px"))
dur2_year_style = widgets.Dropdown(options=DUR2_LINESTYLE_OPTIONS, value="-",
                                    description="Line style:",
                                    style={"description_width": "70px"},
                                    layout=widgets.Layout(width="160px"))
dur2_year_box = widgets.VBox([])  # hidden by default, matching dur2_show_year_toggle=False


def _dur2_toggle_year_box(change):
    dur2_year_box.children = (
        [widgets.HBox([dur2_plot_year, dur2_year_color, dur2_year_style])]
        if change["new"] else []
    )


dur2_show_year_toggle.observe(_dur2_toggle_year_box, names="value")

# -- Number of percentile bands ---------------------------------------------------
dur2_nbands_choice = widgets.Dropdown(
    options=[3, 4, 5, 6, 7], value=7, description="# of bands:",
    style={"description_width": "90px"}, layout=widgets.Layout(width="160px"),
)

dur2_palette_choice = widgets.Dropdown(
    options=DUR2_PALETTE_LABELS,
    value="rainbow", description="Color palette:",
    style={"description_width": "95px"}, layout=widgets.Layout(width="230px"),
)
dur2_palette_swatch = widgets.HTML(
    value=dur2_palette_swatch_html(dur2_palette_choice.value, dur2_nbands_choice.value)
)


def _dur2_on_palette_or_bands_change(change):
    dur2_palette_swatch.value = dur2_palette_swatch_html(
        dur2_palette_choice.value, dur2_nbands_choice.value
    )


dur2_palette_choice.observe(_dur2_on_palette_or_bands_change, names="value")
dur2_nbands_choice.observe(_dur2_on_palette_or_bands_change, names="value")

dur2_log_toggle = widgets.Checkbox(value=True, description="Log scale (y-axis)",
                                    style={"description_width": "initial"})
dur2_median_toggle = widgets.Checkbox(value=True, description="Show median (P50) line",
                                       style={"description_width": "initial"})
dur2_median_color = widgets.ColorPicker(value="#FFFFFF", description="Color:",
                                         style={"description_width": "50px"},
                                         layout=widgets.Layout(width="160px"))
dur2_median_style = widgets.Dropdown(options=DUR2_LINESTYLE_OPTIONS, value="--",
                                      description="Line style:",
                                      style={"description_width": "70px"},
                                      layout=widgets.Layout(width="160px"))

dur2_monthly_avg_toggle = widgets.Checkbox(value=False, description="Show monthly average line",
                                            style={"description_width": "initial"})
dur2_monthly_avg_color = widgets.ColorPicker(value="#000000", description="Color:",
                                              style={"description_width": "50px"},
                                              layout=widgets.Layout(width="160px"))
dur2_monthly_avg_style = widgets.Dropdown(options=DUR2_LINESTYLE_OPTIONS, value="-.",
                                           description="Line style:",
                                           style={"description_width": "70px"},
                                           layout=widgets.Layout(width="160px"))

# -- Optional historical water-year range (hidden unless checked) ---------------
dur2_custom_years_toggle = widgets.Checkbox(
    value=False,
    description="Specify water years for historical percentiles (default: full overlapping record)",
    style={"description_width": "initial"},
)
dur2_hist_start_wy = widgets.BoundedIntText(
    value=1990, min=1900, max=_dur2_today.year + 1, description="History start (WY):",
    style={"description_width": "140px"}, layout=widgets.Layout(width="260px"),
)
dur2_hist_end_wy = widgets.BoundedIntText(
    value=_dur2_today.year, min=1900, max=_dur2_today.year + 1, description="History end (WY):",
    style={"description_width": "140px"}, layout=widgets.Layout(width="260px"),
)
dur2_hist_range_box = widgets.VBox([])


def _dur2_toggle_hist_range(change):
    dur2_hist_range_box.children = (
        [widgets.HBox([dur2_hist_start_wy, dur2_hist_end_wy])] if change["new"] else []
    )


dur2_custom_years_toggle.observe(_dur2_toggle_hist_range, names="value")

dur2_title_input = widgets.Text(value="", description="Title (opt):",
                                 placeholder="e.g. Combined Streamflow Percentiles",
                                 style={"description_width": "90px"},
                                 layout=widgets.Layout(width="500px"))

dur2_run_button = widgets.Button(description="Run", button_style="primary",
                                  layout=widgets.Layout(width="90px"))
dur2_save_button = widgets.Button(description="Save SVG", button_style="success",
                                   layout=widgets.Layout(width="100px"))
dur2_status = widgets.Label(value="")
dur2_out = widgets.Output()

dur2_ui = widgets.VBox([
    widgets.HTML(
        "<div style='display:flex;align-items:center;gap:14px;"
        "background:white;padding:10px 20px;border-radius:6px;"
        f"border:2px solid {WRA_NAVY};margin-bottom:6px'>"
        f"<img src='data:image/png;base64,{_LOGO_B64}' "
        "style='height:42px;width:auto;display:block' alt='WRA logo'>"
        f"<span style='color:{WRA_NAVY};font-size:16px;"
        "font-family:Verdana;font-weight:bold'>"
        "Streamflow Duration Hydrograph — Combined Two-Station Percentiles</span></div>"
    ),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([dur2_site1_id, dur2_site2_id]),
    widgets.HBox([dur2_label]),
    widgets.HBox([dur2_year_type]),
    dur2_show_year_toggle,
    dur2_year_box,
    widgets.HBox([dur2_nbands_choice, dur2_palette_choice, dur2_palette_swatch]),
    widgets.HBox([dur2_log_toggle]),
    widgets.HBox([dur2_median_toggle, dur2_median_color, dur2_median_style]),
    widgets.HBox([dur2_monthly_avg_toggle, dur2_monthly_avg_color, dur2_monthly_avg_style]),
    dur2_custom_years_toggle,
    dur2_hist_range_box,
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([dur2_title_input]),
    widgets.HBox([dur2_run_button, dur2_save_button, dur2_status]),
    dur2_out,
])

_dur2_current_fig = None


# -- Core plot ----------------------------------------------------------------------
def dur2_make_plot(pctl_aligned, flow_aligned, combined_label, year_type, plot_year,
                    use_log, palette_name, show_median, n_bands, custom_title="",
                    median_color="#FFFFFF", median_style="--",
                    year_color="#000000", year_style="-",
                    show_monthly_avg=False, monthly_avg_color="#000000",
                    monthly_avg_style="-.", monthly_avg_series=None):
    global _dur2_current_fig

    colors = dur2_get_palette_colors(palette_name, n_bands)
    band_defs = dur2_band_defs(n_bands)

    fig, ax = plt.subplots(figsize=(13, 6))

    for (lower, upper, _label), color in zip(band_defs, colors):
        ax.fill_between(pctl_aligned["plot_date"], pctl_aligned[lower],
                         pctl_aligned[upper], color=color, alpha=0.85,
                         linewidth=0, zorder=1)

    if show_median:
        ax.plot(pctl_aligned["plot_date"], pctl_aligned["p50_va"],
                color=median_color, linewidth=1.0, linestyle=median_style, alpha=0.8,
                zorder=2, label="Median (P50)")

    if show_monthly_avg and monthly_avg_series is not None:
        ax.plot(monthly_avg_series["plot_date"], monthly_avg_series["avg"],
                color=monthly_avg_color, linewidth=1.6, linestyle=monthly_avg_style,
                marker="o", markersize=3, zorder=4, label="Monthly average")

    if flow_aligned is not None:
        yr_label = f"Water Year {plot_year}" if year_type == "water" else f"{plot_year}"
        ax.plot(flow_aligned["plot_date"], flow_aligned["flow_cfs"],
                color=year_color, linewidth=1.8, linestyle=year_style,
                zorder=5, label=f"{yr_label} combined daily mean")

    if use_log:
        ax.set_yscale("log")

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.set_xlim(pctl_aligned["plot_date"].min(), pctl_aligned["plot_date"].max())
    ax.set_xlabel("Month")
    ax.set_ylabel("Combined Discharge (cfs)")

    title_str = (custom_title.strip() if custom_title.strip() else
                 f"Combined Streamflow Percentiles by Day of Year - {combined_label}")
    ax.set_title(title_str)
    ax.grid(True, which="both", alpha=0.25)

    band_handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.85) for c in colors]
    band_labels = [lbl for _, _, lbl in band_defs]
    line_handles, line_labels = ax.get_legend_handles_labels()
    ax.legend(list(reversed(band_handles)) + line_handles,
              list(reversed(band_labels)) + line_labels,
              loc="upper right", fontsize=7, framealpha=0.9, ncol=1)

    fig.text(0.99, 0.01, f"© {dt.date.today().year} WRA, Inc.",
              ha="right", va="bottom", fontsize=7,
              fontfamily="Verdana", color="#888888")

    fig.tight_layout()
    _dur2_current_fig = fig
    plt.show()


# -- Button callbacks -----------------------------------------------------------------
def dur2_on_run(b):
    with dur2_out:
        clear_output(wait=True)
        try:
            site1 = dur2_site1_id.value.strip()
            site2 = dur2_site2_id.value.strip()
            if not site1 or not site2:
                dur2_status.value = "Please enter both Station 1 and Station 2 IDs."
                return

            year_type = dur2_year_type.value
            show_year = dur2_show_year_toggle.value
            n_bands = int(dur2_nbands_choice.value)

            if show_year:
                plot_year = int(dur2_plot_year.value)
            else:
                # No overlay year requested -- still need a reference year to
                # lay out the x-axis (Oct-Sep vs Jan-Dec), just not shown.
                plot_year = (dur2_water_year_of(_dur2_today)
                             if year_type == "water" else _dur2_today.year)

            dur2_status.value = f"Fetching full daily history for Station {site1}..."
            full1 = dur2_fetch_full_daily_flow(site1)

            dur2_status.value = f"Fetching full daily history for Station {site2}..."
            full2 = dur2_fetch_full_daily_flow(site2)

            dur2_status.value = "Combining both stations' daily flow..."
            combined = dur2_combine_two_stations(full1, full2)

            if dur2_custom_years_toggle.value:
                hist_start_wy = int(dur2_hist_start_wy.value)
                hist_end_wy = int(dur2_hist_end_wy.value)
                if hist_start_wy > hist_end_wy:
                    raise ValueError("History start (WY) must be <= History end (WY).")
            else:
                hist_start_wy = None
                hist_end_wy = None

            dur2_status.value = "Computing combined percentiles..."
            pctl, rec_start_wy, rec_end_wy = dur2_compute_percentiles(
                combined, hist_start_wy, hist_end_wy
            )
            pctl["plot_date"] = pctl.apply(
                lambda r: dur2_plot_date(r["month_nu"], r["day_nu"], plot_year, year_type),
                axis=1,
            )
            pctl["plot_date"] = pd.to_datetime(pctl["plot_date"])
            pctl = pctl.sort_values("plot_date")

            flow_year = None
            if show_year:
                # Selected year's combined daily-mean trace (pulled straight
                # out of the already-fetched combined series).
                if year_type == "water":
                    yr_start = dt.date(plot_year - 1, 10, 1)
                    yr_end = dt.date(plot_year, 9, 30)
                else:
                    yr_start = dt.date(plot_year, 1, 1)
                    yr_end = dt.date(plot_year, 12, 31)

                flow_year = combined[
                    (combined["date"].dt.date >= yr_start)
                    & (combined["date"].dt.date <= yr_end)
                ].copy()
                if flow_year.empty:
                    yt_label = "water year" if year_type == "water" else "calendar year"
                    raise ValueError(f"No combined data available for {yt_label} {plot_year}.")
                flow_year["plot_date"] = flow_year["date"].apply(
                    lambda d: dur2_plot_date(d.month, d.day, plot_year, year_type)
                )
                flow_year["plot_date"] = pd.to_datetime(flow_year["plot_date"])
                flow_year = flow_year.sort_values("plot_date")

            name1 = dur2_fetch_station_name(site1)
            name2 = dur2_fetch_station_name(site2)
            combined_label = dur2_label.value.strip() or f"{name1} + {name2}"

            combined["water_year"] = combined["date"].apply(dur2_water_year_of)
            if hist_start_wy is not None:
                table_data = combined[
                    (combined["water_year"] >= hist_start_wy)
                    & (combined["water_year"] <= hist_end_wy)
                ]
            else:
                table_data = combined

            monthly_avg_series = None
            if dur2_monthly_avg_toggle.value:
                monthly_avg_series = dur2_build_monthly_avg_series(table_data, plot_year, year_type)

            dur2_status.value = "Rendering chart..."
            dur2_make_plot(pctl, flow_year, combined_label, year_type, plot_year,
                            dur2_log_toggle.value, dur2_palette_choice.value,
                            dur2_median_toggle.value, n_bands, dur2_title_input.value,
                            median_color=dur2_median_color.value, median_style=dur2_median_style.value,
                            year_color=dur2_year_color.value, year_style=dur2_year_style.value,
                            show_monthly_avg=dur2_monthly_avg_toggle.value,
                            monthly_avg_color=dur2_monthly_avg_color.value,
                            monthly_avg_style=dur2_monthly_avg_style.value,
                            monthly_avg_series=monthly_avg_series)

            total1 = dur2_fetch_period_of_record(site1)
            total2 = dur2_fetch_period_of_record(site2)
            print(f"\n{combined_label}")
            display(dur2_build_summary_table(table_data, site1, site2, total1, total2,
                                              rec_start_wy, rec_end_wy))

            dur2_status.value = (
                f"Done - {combined_label} "
                f"(combined overlapping record: WY{rec_start_wy}-WY{rec_end_wy})"
            )

        except Exception as e:
            dur2_status.value = f"Error: {e}"
            raise


def dur2_on_save(b):
    global _dur2_current_fig
    if _dur2_current_fig is None:
        dur2_status.value = "Run the plot first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save combined duration hydrograph as SVG",
            defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            _dur2_current_fig.savefig(path, format="svg", bbox_inches="tight")
            dur2_status.value = f"Saved: {path}"
        else:
            dur2_status.value = "Save cancelled."
    except Exception as e:
        dur2_status.value = f"Save error: {e}"


dur2_run_button.on_click(dur2_on_run)
dur2_save_button.on_click(dur2_on_save)

# -- Launch -------------------------------------------------------------------------
display(dur2_ui)

In [10]:
# =============================================================================
#  WRA Streamflow Analysis — Weibull Flow-Duration / Exceedance Probability
#
#  Fetches the FULL daily-mean-flow history for a station, lets the user
#  restrict to a water-year range and to specific calendar months, then runs
#  a Weibull plotting-position analysis on the retained daily values:
#
#       P_exceed (%) = 100 * m / (n + 1)
#
#  where m is the rank of a flow value (m = 1 for the highest flow) and n is
#  the number of retained daily values.
#
#  A SECOND station can optionally be overlaid on the same plot as its own
#  independent flow-duration curve (a different color/line style), for
#  side-by-side comparison. The two stations are NOT combined/summed here --
#  each gets its own Weibull analysis, computed and plotted separately.
#
#  Output: a Flow (y-axis, log-optional) vs. Percent of Time Equaled or
#  Exceeded (x-axis, 0-100%) plot, plus an interpolated summary table at
#  standard exceedance percentages for each station shown, and a CSV export
#  of the full point set(s).
# =============================================================================

import datetime as dt
import calendar

import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import rcParams

import ipywidgets as widgets
from IPython.display import display, clear_output

import tkinter as tk
from tkinter import filedialog

# -----------------------------------------------------------------------------
# WRA branding — paste the same base64 logo string used in your other tools
# here (the constant is called _LOGO_B64 in your duration-hydrograph tools).
# Leaving it blank simply omits the logo image from the header banner.
# -----------------------------------------------------------------------------
_LOGO_B64 = "..."

WRA_NAVY = "#175536"
WRA_GOLD = "#C9A84C"

rcParams["font.family"] = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"] = 13
rcParams["axes.labelsize"] = 11
rcParams["xtick.labelsize"] = 8
rcParams["ytick.labelsize"] = 8

WFD_LINESTYLE_OPTIONS = [("Solid", "-"), ("Dashed", "--"), ("Dash-dot", "-."), ("Dotted", ":")]

# Standard percent-exceedance thresholds used for the interpolated summary table.
WFD_STANDARD_PROBS = [0.1, 1, 2, 5, 10, 20, 25, 30, 40, 50,
                       60, 70, 75, 80, 90, 95, 98, 99, 99.9]


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def wfd_water_year_of(date):
    """Water year = calendar year of the following Jan-Sep, i.e. Oct-Dec
    counts toward the NEXT calendar year's water year."""
    return date.year + 1 if date.month >= 10 else date.year


def wfd_fetch_full_daily_flow(site_no, start_date="1890-01-01", end_date=None):
    """Pull the FULL available daily-mean-flow (parameter 00060, stat 00003)
    history for one USGS station."""
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "json",
        "sites": site_no,
        "startDT": start_date,
        "parameterCd": "00060",
        "statCd": "00003",
    }
    if end_date:
        params["endDT"] = end_date
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    ts = data.get("value", {}).get("timeSeries", [])
    if not ts:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    values = ts[0]["values"][0]["value"]
    df = pd.DataFrame(values)
    if df.empty:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
    df["flow_cfs"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["flow_cfs"])
    return df.rename(columns={"dateTime": "date"})[["date", "flow_cfs"]].sort_values("date")


def wfd_fetch_station_name(site_no):
    """Best-effort station name lookup; falls back to a generic label."""
    url = "https://waterservices.usgs.gov/nwis/site/"
    params = {"sites": site_no, "format": "rdb", "siteOutput": "expanded"}
    try:
        r = requests.get(url, params=params, timeout=15)
        for line in r.text.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) > 2 and parts[0] == site_no:
                return parts[2].strip().title()
    except Exception:
        pass
    return f"USGS Station {site_no}"


def wfd_filter_by_years_months(df, start_wy, end_wy, months):
    """Restrict a daily-flow dataframe to a water-year range and a set of
    calendar months (1-12)."""
    d = df.copy()
    d["water_year"] = d["date"].apply(wfd_water_year_of)
    d = d[(d["water_year"] >= start_wy) & (d["water_year"] <= end_wy)]
    d = d[d["date"].dt.month.isin(months)]
    return d.dropna(subset=["flow_cfs"])


def wfd_compute_weibull_table(df):
    """Sort flows descending (rank 1 = highest flow) and assign the Weibull
    plotting-position exceedance probability to every retained daily value."""
    d = df.dropna(subset=["flow_cfs"]).copy()
    d = d.sort_values("flow_cfs", ascending=False).reset_index(drop=True)
    n = len(d)
    if n == 0:
        raise ValueError("No data available after filtering by water year(s) and month(s).")
    d["rank"] = np.arange(1, n + 1)
    d["exceedance_prob_pct"] = 100.0 * d["rank"] / (n + 1.0)
    return d


def wfd_summary_table(weibull_df, probs=None):
    """Interpolate the flow value at standard exceedance-probability
    thresholds from the full Weibull point set."""
    if probs is None:
        probs = WFD_STANDARD_PROBS
    x = weibull_df["exceedance_prob_pct"].values  # increases with rank
    y = weibull_df["flow_cfs"].values              # decreases with rank
    flows = np.interp(probs, x, y)
    out = pd.DataFrame({
        "Percent of Time Equaled or Exceeded (%)": probs,
        "Flow (cfs)": np.round(flows, 1),
    })
    return out


# -----------------------------------------------------------------------------
# Widgets
# -----------------------------------------------------------------------------
wfd_site1_id = widgets.Text(
    value="", description="Station 1 ID:", placeholder="e.g. 11169500",
    style={"description_width": "100px"}, layout=widgets.Layout(width="260px"),
)

wfd_compare_toggle = widgets.Checkbox(
    value=False, description="Overlay a second station (separate curve, not combined)",
    style={"description_width": "initial"},
)
wfd_site2_id = widgets.Text(
    value="", description="Station 2 ID:", placeholder="e.g. 11169800",
    style={"description_width": "100px"}, layout=widgets.Layout(width="260px"),
)
wfd_site2_box = widgets.HBox([])  # hidden until wfd_compare_toggle is checked


def _wfd_toggle_site2_box(change):
    wfd_site2_box.children = [wfd_site2_id] if change["new"] else []


wfd_compare_toggle.observe(_wfd_toggle_site2_box, names="value")

wfd_fetch_button = widgets.Button(description="Fetch Station Data", button_style="info",
                                   layout=widgets.Layout(width="170px"))
wfd_fetch_status = widgets.Label(value="")

# Water-year range slider (populated once data is fetched; spans whichever
# station(s) are loaded so either curve can use the full range it has data for)
_wfd_today = dt.date.today()
wfd_year_slider = widgets.IntRangeSlider(
    value=[1990, _wfd_today.year], min=1900, max=_wfd_today.year + 1, step=1,
    description="Water years:", continuous_update=False, disabled=True,
    style={"description_width": "90px"}, layout=widgets.Layout(width="480px"),
)

# Month checkboxes (Jan-Dec), 4 columns x 3 rows
wfd_month_checks = {
    m: widgets.Checkbox(value=True, description=calendar.month_abbr[m], indent=False,
                         layout=widgets.Layout(width="90px"))
    for m in range(1, 13)
}
wfd_month_grid = widgets.GridBox(
    children=[wfd_month_checks[m] for m in range(1, 13)],
    layout=widgets.Layout(grid_template_columns="repeat(4, 100px)"),
)
wfd_select_all_months = widgets.Button(description="Select All", layout=widgets.Layout(width="100px"))
wfd_clear_all_months = widgets.Button(description="Clear All", layout=widgets.Layout(width="100px"))


def _wfd_on_select_all_months(b):
    for cb in wfd_month_checks.values():
        cb.value = True


def _wfd_on_clear_all_months(b):
    for cb in wfd_month_checks.values():
        cb.value = False


wfd_select_all_months.on_click(_wfd_on_select_all_months)
wfd_clear_all_months.on_click(_wfd_on_clear_all_months)

wfd_log_toggle = widgets.Checkbox(value=True, description="Log scale (y-axis)",
                                   style={"description_width": "initial"})
wfd_show_points_toggle = widgets.Checkbox(value=False, description="Show individual daily points",
                                           style={"description_width": "initial"})
wfd_show_markers_toggle = widgets.Checkbox(value=True, description="Mark standard exceedance percentages",
                                            style={"description_width": "initial"})

wfd_curve1_color = widgets.ColorPicker(value=WRA_NAVY, description="Station 1 color:",
                                        style={"description_width": "100px"},
                                        layout=widgets.Layout(width="230px"))
wfd_curve1_style = widgets.Dropdown(options=WFD_LINESTYLE_OPTIONS, value="-",
                                     description="Line style:",
                                     style={"description_width": "70px"},
                                     layout=widgets.Layout(width="160px"))
wfd_curve2_color = widgets.ColorPicker(value="#B22222", description="Station 2 color:",
                                        style={"description_width": "100px"},
                                        layout=widgets.Layout(width="230px"))
wfd_curve2_style = widgets.Dropdown(options=WFD_LINESTYLE_OPTIONS, value="--",
                                     description="Line style:",
                                     style={"description_width": "70px"},
                                     layout=widgets.Layout(width="160px"))
wfd_curve2_style_box = widgets.HBox([])  # hidden until wfd_compare_toggle is checked


def _wfd_toggle_curve2_style_box(change):
    wfd_curve2_style_box.children = (
        [wfd_curve2_color, wfd_curve2_style] if change["new"] else []
    )


wfd_compare_toggle.observe(_wfd_toggle_curve2_style_box, names="value")

wfd_title_input = widgets.Text(
    value="", description="Title (opt):",
    placeholder="e.g. Flow-Duration Curve — Station 11169500",
    style={"description_width": "90px"}, layout=widgets.Layout(width="500px"),
)

wfd_run_button = widgets.Button(description="Run Analysis", button_style="primary",
                                 layout=widgets.Layout(width="130px"))
wfd_save_svg_button = widgets.Button(description="Save Plot SVG", button_style="success",
                                      layout=widgets.Layout(width="130px"))
wfd_save_csv_button = widgets.Button(description="Save Full Table CSV", button_style="success",
                                      layout=widgets.Layout(width="160px"))
wfd_status = widgets.Label(value="")
wfd_out = widgets.Output()

_wfd_header_html = (
    "<div style='display:flex;align-items:center;gap:14px;"
    "background:white;padding:10px 20px;border-radius:6px;"
    f"border:2px solid {WRA_NAVY};margin-bottom:6px'>"
    + (f"<img src='data:image/png;base64,{_LOGO_B64}' "
       "style='height:42px;width:auto;display:block' alt='WRA logo'>"
       if _LOGO_B64 else "")
    + f"<span style='color:{WRA_NAVY};font-size:16px;"
      "font-family:Verdana;font-weight:bold'>"
      "Weibull Flow-Duration Analysis — Flow vs. Exceedance Probability</span></div>"
)

wfd_ui = widgets.VBox([
    widgets.HTML(_wfd_header_html),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([wfd_site1_id, wfd_fetch_button, wfd_fetch_status]),
    wfd_compare_toggle,
    wfd_site2_box,
    widgets.HTML("<b>Water years to include:</b>"),
    wfd_year_slider,
    widgets.HTML("<b>Months to include:</b>"),
    wfd_month_grid,
    widgets.HBox([wfd_select_all_months, wfd_clear_all_months]),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([wfd_log_toggle, wfd_show_points_toggle, wfd_show_markers_toggle]),
    widgets.HBox([wfd_curve1_color, wfd_curve1_style]),
    wfd_curve2_style_box,
    widgets.HBox([wfd_title_input]),
    widgets.HBox([wfd_run_button, wfd_save_svg_button, wfd_save_csv_button, wfd_status]),
    wfd_out,
])

# -----------------------------------------------------------------------------
# State held between callbacks (fetched data, computed tables, current figure)
# -----------------------------------------------------------------------------
_wfd_state = {
    "full_df1": None,
    "site1": None,
    "name1": None,
    "full_df2": None,
    "site2": None,
    "name2": None,
    "weibull1": None,
    "weibull2": None,
    "current_fig": None,
}


# -----------------------------------------------------------------------------
# Fetch callback
# -----------------------------------------------------------------------------
def wfd_on_fetch(b):
    site1 = wfd_site1_id.value.strip()
    if not site1:
        wfd_fetch_status.value = "Enter a Station 1 ID first."
        return
    try:
        wfd_fetch_status.value = f"Fetching full daily history for Station {site1}..."
        full_df1 = wfd_fetch_full_daily_flow(site1)
        name1 = wfd_fetch_station_name(site1)

        full_df2, site2, name2 = None, None, None
        if wfd_compare_toggle.value:
            site2 = wfd_site2_id.value.strip()
            if site2:
                wfd_fetch_status.value = f"Fetching full daily history for Station {site2}..."
                full_df2 = wfd_fetch_full_daily_flow(site2)
                name2 = wfd_fetch_station_name(site2)
            else:
                wfd_fetch_status.value = (
                    "Overlay is checked but Station 2 ID is blank -- "
                    "loaded Station 1 only."
                )

        # Slider spans the full range covered by whichever station(s) loaded,
        # so each curve's own filter naturally limits itself to real data.
        all_dates = full_df1["date"]
        if full_df2 is not None:
            all_dates = pd.concat([all_dates, full_df2["date"]])
        wy_all = all_dates.apply(wfd_water_year_of)
        wy_min, wy_max = int(wy_all.min()), int(wy_all.max())

        wfd_year_slider.min = wy_min
        wfd_year_slider.max = wy_max
        wfd_year_slider.value = [wy_min, wy_max]
        wfd_year_slider.disabled = False

        _wfd_state["full_df1"] = full_df1
        _wfd_state["site1"] = site1
        _wfd_state["name1"] = name1
        _wfd_state["full_df2"] = full_df2
        _wfd_state["site2"] = site2
        _wfd_state["name2"] = name2

        if full_df2 is not None:
            wfd_fetch_status.value = (
                f"Loaded {name1} ({site1}) and {name2} ({site2}) — "
                f"WY{wy_min}-WY{wy_max} combined range."
            )
        else:
            wfd_fetch_status.value = (
                f"Loaded {name1} ({site1}) — WY{wy_min}-WY{wy_max}, "
                f"{len(full_df1):,} daily values."
            )
    except Exception as e:
        wfd_fetch_status.value = f"Error: {e}"
        raise


wfd_fetch_button.on_click(wfd_on_fetch)


# -----------------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------------
def wfd_make_plot(weibull1, label1, curve1_color, curve1_style,
                   weibull2, label2, curve2_color, curve2_style,
                   start_wy, end_wy, months, use_log, show_points, show_markers,
                   custom_title=""):
    fig, ax = plt.subplots(figsize=(11, 6.5))

    points_labeled = False

    if show_points:
        ax.scatter(weibull1["exceedance_prob_pct"], weibull1["flow_cfs"],
                   s=6, color="#AAAAAA", alpha=0.5, zorder=1,
                   label="Individual daily values")
        points_labeled = True
        if weibull2 is not None:
            ax.scatter(weibull2["exceedance_prob_pct"], weibull2["flow_cfs"],
                       s=6, color="#AAAAAA", alpha=0.5, zorder=1,
                       label=None if points_labeled else "Individual daily values")

    ax.plot(weibull1["exceedance_prob_pct"], weibull1["flow_cfs"],
            color=curve1_color, linewidth=1.8, linestyle=curve1_style,
            zorder=2, label=label1)

    if weibull2 is not None:
        ax.plot(weibull2["exceedance_prob_pct"], weibull2["flow_cfs"],
                color=curve2_color, linewidth=1.8, linestyle=curve2_style,
                zorder=2, label=label2)

    if show_markers:
        summary1 = wfd_summary_table(weibull1, probs=[5, 10, 25, 50, 75, 90, 95])
        ax.scatter(summary1["Percent of Time Equaled or Exceeded (%)"],
                   summary1["Flow (cfs)"], color=WRA_GOLD, edgecolor=curve1_color,
                   s=40, zorder=3, label="Standard exceedance percentiles")
        if weibull2 is not None:
            summary2 = wfd_summary_table(weibull2, probs=[5, 10, 25, 50, 75, 90, 95])
            ax.scatter(summary2["Percent of Time Equaled or Exceeded (%)"],
                       summary2["Flow (cfs)"], color="white", edgecolor=curve2_color,
                       s=40, zorder=3, label=None)

    if use_log:
        ax.set_yscale("log")

    ax.set_xlim(0, 100)
    ax.set_xlabel("Percent of Time Flow Was Equaled or Exceeded (%)")
    ax.set_ylabel("Flow (cfs)")

    month_abbrs = ", ".join(calendar.month_abbr[m] for m in months)
    if weibull2 is not None:
        default_title = (
            f"Flow-Duration Curve Comparison (Weibull Plotting Position)\n"
            f"WY{start_wy}-WY{end_wy} — Months: {month_abbrs}"
        )
    else:
        default_title = (
            f"Flow-Duration Curve (Weibull Plotting Position)\n"
            f"{label1} — WY{start_wy}-WY{end_wy} — Months: {month_abbrs}"
        )
    ax.set_title(custom_title.strip() if custom_title.strip() else default_title, fontsize=11)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)

    fig.text(0.99, 0.01, f"© {dt.date.today().year} WRA, Inc.",
              ha="right", va="bottom", fontsize=7,
              fontfamily="Verdana", color="#888888")

    fig.tight_layout()
    _wfd_state["current_fig"] = fig
    plt.show()


# -----------------------------------------------------------------------------
# Run callback
# -----------------------------------------------------------------------------
def wfd_on_run(b):
    with wfd_out:
        clear_output(wait=True)
        try:
            full_df1 = _wfd_state["full_df1"]
            if full_df1 is None:
                wfd_status.value = "Fetch station data first."
                return

            start_wy, end_wy = wfd_year_slider.value
            months = [m for m, cb in wfd_month_checks.items() if cb.value]
            if not months:
                wfd_status.value = "Select at least one month."
                return

            wfd_status.value = "Filtering Station 1 data..."
            filtered1 = wfd_filter_by_years_months(full_df1, start_wy, end_wy, months)
            if filtered1.empty:
                raise ValueError(
                    "No Station 1 daily values fall within the selected "
                    "water-year range and month(s)."
                )
            weibull1 = wfd_compute_weibull_table(filtered1)
            _wfd_state["weibull1"] = weibull1

            name1, site1 = _wfd_state["name1"], _wfd_state["site1"]
            label1 = f"{name1} ({site1})"

            weibull2, label2 = None, None
            full_df2 = _wfd_state["full_df2"]
            if wfd_compare_toggle.value and full_df2 is not None:
                wfd_status.value = "Filtering Station 2 data..."
                filtered2 = wfd_filter_by_years_months(full_df2, start_wy, end_wy, months)
                if filtered2.empty:
                    wfd_status.value = (
                        "Station 2 has no data in the selected range/months -- "
                        "showing Station 1 only."
                    )
                else:
                    weibull2 = wfd_compute_weibull_table(filtered2)
                    name2, site2 = _wfd_state["name2"], _wfd_state["site2"]
                    label2 = f"{name2} ({site2})"
            _wfd_state["weibull2"] = weibull2

            wfd_status.value = "Rendering chart..."
            wfd_make_plot(
                weibull1, label1, wfd_curve1_color.value, wfd_curve1_style.value,
                weibull2, label2, wfd_curve2_color.value, wfd_curve2_style.value,
                start_wy, end_wy, months, wfd_log_toggle.value,
                wfd_show_points_toggle.value, wfd_show_markers_toggle.value,
                wfd_title_input.value,
            )

            summary1 = wfd_summary_table(weibull1)
            print(f"\n{label1} — WY{start_wy}-WY{end_wy} — "
                  f"n = {len(weibull1):,} daily values")
            print("Flow-Duration Summary Table (Weibull plotting position, interpolated):\n")
            display(summary1)

            if weibull2 is not None:
                summary2 = wfd_summary_table(weibull2)
                print(f"\n{label2} — WY{start_wy}-WY{end_wy} — "
                      f"n = {len(weibull2):,} daily values")
                print("Flow-Duration Summary Table (Weibull plotting position, interpolated):\n")
                display(summary2)

            wfd_status.value = "Done. Use 'Save Full Table CSV' to export every point."
        except Exception as e:
            wfd_status.value = f"Error: {e}"
            raise


wfd_run_button.on_click(wfd_on_run)


# -----------------------------------------------------------------------------
# Save callbacks
# -----------------------------------------------------------------------------
def wfd_on_save_svg(b):
    fig = _wfd_state["current_fig"]
    if fig is None:
        wfd_status.value = "Run the analysis first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save flow-duration curve as SVG",
            defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            fig.savefig(path, format="svg", bbox_inches="tight")
            wfd_status.value = f"Saved plot: {path}"
        else:
            wfd_status.value = "Save cancelled."
    except Exception as e:
        wfd_status.value = f"Save error: {e}"


def wfd_on_save_csv(b):
    weibull1 = _wfd_state["weibull1"]
    if weibull1 is None:
        wfd_status.value = "Run the analysis first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save full Weibull flow-duration table as CSV",
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            out_cols = ["date", "flow_cfs", "rank", "exceedance_prob_pct"]
            t1 = weibull1[out_cols].copy()
            t1.insert(0, "station", _wfd_state["site1"])

            weibull2 = _wfd_state["weibull2"]
            if weibull2 is not None:
                t2 = weibull2[out_cols].copy()
                t2.insert(0, "station", _wfd_state["site2"])
                combined_out = pd.concat([t1, t2], ignore_index=True)
            else:
                combined_out = t1

            combined_out.to_csv(path, index=False)
            wfd_status.value = f"Saved full table ({len(combined_out):,} rows): {path}"
        else:
            wfd_status.value = "Save cancelled."
    except Exception as e:
        wfd_status.value = f"Save error: {e}"


wfd_save_svg_button.on_click(wfd_on_save_svg)
wfd_save_csv_button.on_click(wfd_on_save_csv)

# -----------------------------------------------------------------------------
# Launch
# -----------------------------------------------------------------------------
display(wfd_ui)

In [13]:
# =============================================================================
#  WRA Streamflow Analysis — Weibull Flow-Duration / Exceedance Probability
#  COMBINED TWO-STATION VERSION (water-year slider + recurring month/day window)
#
#  Fetches full daily-mean-flow history for TWO USGS stations, sums their
#  flows on matching dates (inner join, same approach as the combined
#  duration-hydrograph tool), lets the user:
#
#    1. Pick a WATER-YEAR RANGE with a slider (e.g. WY1990-WY2020), and
#    2. Pick a recurring MONTH/DAY WINDOW (no year) -- e.g. Jun 15 to Sep 15
#       -- that gets applied inside every one of those water years.
#
#  ...then runs a Weibull plotting-position analysis on the retained
#  COMBINED daily values:
#
#       P_exceed (%) = 100 * m / (n + 1)
#       Return Period (years) = 100 / P_exceed (%)
#
#  where m is the rank of a flow value (m = 1 for the highest flow) and n is
#  the number of retained daily values.
#
#  A month/day window that starts in Oct-Dec and ends in Jan-Sep (e.g. Nov 1
#  to Feb 28) is treated as spanning the turn of the calendar year WITHIN a
#  single water year -- e.g. for WY2020 that means Nov 1, 2019 to Feb 28, 2020.
#
#  NOTE on "return period": this is computed directly from the day-based
#  Weibull exceedance percentage of the selected years/season. It describes
#  how rare a given DAILY flow is within that selection -- it is NOT the same
#  as a flood-frequency return period derived from annual peak flows (e.g.
#  via LP3/EMA on annual maxima). Keep that distinction in mind for reports.
#
#  Output: a Flow (y-axis, log-optional) vs. Percent of Time Equaled or
#  Exceeded (x-axis, 0-100%) plot, a summary table (Return Period, Percent
#  Exceeded, Flow columns, plus an info block showing the water years and
#  month/day window actually analyzed), a CSV export of the full point set,
#  and a user-adjustable dashed threshold line (default 50%) with a callout
#  box showing the interpolated flow value at that percentage.
# =============================================================================

import datetime as dt
import calendar

import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import rcParams

import ipywidgets as widgets
from IPython.display import display, clear_output

import tkinter as tk
from tkinter import filedialog

# -----------------------------------------------------------------------------
# WRA branding — paste the same base64 logo string used in your other tools
# here (the constant is called _LOGO_B64 in your duration-hydrograph tools).
# Leaving it blank simply omits the logo image from the header banner.
# -----------------------------------------------------------------------------
_LOGO_B64 = "..."

WRA_NAVY = "#175536"
WRA_GOLD = "#C9A84C"

rcParams["font.family"] = "Verdana"
rcParams["font.sans-serif"] = ["Verdana"]
rcParams["axes.titlesize"] = 13
rcParams["axes.labelsize"] = 11
rcParams["xtick.labelsize"] = 8
rcParams["ytick.labelsize"] = 8

WFC_LINESTYLE_OPTIONS = [("Solid", "-"), ("Dashed", "--"), ("Dash-dot", "-."), ("Dotted", ":")]

# Standard return periods (years) shown in the summary table -- classic
# flood/duration-reporting values, from very frequent to very rare.
WFC_TABLE_RETURN_PERIODS = [1.01, 1.05, 1.1, 1.25, 1.5, 2, 5, 10, 25, 50, 100, 200, 500]

# A smaller subset used for the gold markers plotted directly on the curve
# (keeps the chart readable).
WFC_PLOT_MARKER_RETURN_PERIODS = [2, 5, 10, 25, 50, 100]

WFC_MONTH_OPTIONS = [(calendar.month_abbr[m], m) for m in range(1, 13)]
WFC_DAY_OPTIONS = list(range(1, 32))


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def wfc_water_year_of(date):
    """Water year = calendar year of the following Jan-Sep, i.e. Oct-Dec
    counts toward the NEXT calendar year's water year."""
    return date.year + 1 if date.month >= 10 else date.year


def wfc_wy_month_index(month):
    """Position of a calendar month within the water-year timeline, where
    Oct=0, Nov=1, ..., Sep=11. Lets us check that a (start_month, end_month)
    pair is in valid chronological order WITHIN a single water year, since
    raw calendar-month numbers don't reflect that Oct-Dec comes before
    Jan-Sep in water-year terms."""
    return (int(month) - 10) % 12


def wfc_calendar_date_for_wy(wy, month, day):
    """Map a (month, day) with no year onto the actual calendar date within
    a given water year -- Oct/Nov/Dec fall in the PRIOR calendar year,
    Jan-Sep fall in the water year's own calendar year. Clamps to the last
    valid day of the month if the day doesn't exist (e.g. Feb 30, or Feb 29
    in a non-leap year)."""
    wy, month, day = int(wy), int(month), int(day)
    calendar_year = wy - 1 if month >= 10 else wy
    last_day = calendar.monthrange(calendar_year, month)[1]
    day = min(day, last_day)
    return dt.date(calendar_year, month, day)


def wfc_fetch_full_daily_flow(site_no, start_date="1890-01-01", end_date=None):
    """Pull the FULL available daily-mean-flow (parameter 00060, stat 00003)
    history for one USGS station."""
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "json",
        "sites": site_no,
        "startDT": start_date,
        "parameterCd": "00060",
        "statCd": "00003",
    }
    if end_date:
        params["endDT"] = end_date
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    ts = data.get("value", {}).get("timeSeries", [])
    if not ts:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    values = ts[0]["values"][0]["value"]
    df = pd.DataFrame(values)
    if df.empty:
        raise ValueError(f"No daily-value data returned for site {site_no}.")
    df["dateTime"] = pd.to_datetime(df["dateTime"]).dt.tz_localize(None)
    df["flow_cfs"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["flow_cfs"])
    return df.rename(columns={"dateTime": "date"})[["date", "flow_cfs"]].sort_values("date")


def wfc_combine_two_stations(df1, df2):
    """Sum two stations' daily flow on matching dates only (inner join) --
    a date only counts if BOTH stations reported a value that day."""
    merged = pd.merge(df1, df2, on="date", suffixes=("_1", "_2"))
    if merged.empty:
        raise ValueError(
            "The two stations have no overlapping dates with data -- "
            "cannot combine them."
        )
    merged["flow_cfs"] = merged["flow_cfs_1"] + merged["flow_cfs_2"]
    return merged[["date", "flow_cfs"]].sort_values("date")


def wfc_fetch_station_name(site_no):
    """Best-effort station name lookup; falls back to a generic label."""
    url = "https://waterservices.usgs.gov/nwis/site/"
    params = {"sites": site_no, "format": "rdb", "siteOutput": "expanded"}
    try:
        r = requests.get(url, params=params, timeout=15)
        for line in r.text.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) > 2 and parts[0] == site_no:
                return parts[2].strip().title()
    except Exception:
        pass
    return f"USGS Station {site_no}"


def wfc_filter_by_years_and_season(df, start_wy, end_wy, start_month, start_day,
                                    end_month, end_day):
    """Restrict a daily-flow dataframe to a water-year range, keeping only
    dates that fall inside the recurring (month, day) window in EACH of
    those water years (the window can cross the calendar-year turn, e.g.
    Nov 1 - Feb 28, and stays correct within a single water year)."""
    start_idx = wfc_wy_month_index(start_month)
    end_idx = wfc_wy_month_index(end_month)
    if start_idx > end_idx or (start_idx == end_idx and start_day > end_day):
        raise ValueError(
            "Start month/day must come before end month/day within a single "
            "water year (Oct 1 - Sep 30 order). For example, Jun 15 -> Sep 15 "
            "is valid; Nov 1 -> Feb 28 is valid (crosses the calendar-year "
            "turn); but Jan 1 -> Dec 31 is NOT valid since Dec falls earlier "
            "in the water year than Jan. Use Oct 1 -> Sep 30 for a full year."
        )

    mask = pd.Series(False, index=df.index)
    for wy in range(int(start_wy), int(end_wy) + 1):
        window_start = wfc_calendar_date_for_wy(wy, start_month, start_day)
        window_end = wfc_calendar_date_for_wy(wy, end_month, end_day)
        start_ts = pd.Timestamp(window_start)
        end_ts = pd.Timestamp(window_end)
        mask = mask | ((df["date"] >= start_ts) & (df["date"] <= end_ts))
    d = df[mask].copy()
    d["water_year"] = d["date"].apply(wfc_water_year_of)
    return d.dropna(subset=["flow_cfs"])


def wfc_compute_weibull_table(df):
    """Sort flows descending (rank 1 = highest flow), assign the Weibull
    plotting-position exceedance probability, and back out the equivalent
    return period (years) for every retained daily value."""
    d = df.dropna(subset=["flow_cfs"]).copy()
    d = d.sort_values("flow_cfs", ascending=False).reset_index(drop=True)
    n = len(d)
    if n == 0:
        raise ValueError(
            "No data available for the selected water years and month/day window."
        )
    d["rank"] = np.arange(1, n + 1)
    d["exceedance_prob_pct"] = 100.0 * d["rank"] / (n + 1.0)
    d["return_period_years"] = 100.0 / d["exceedance_prob_pct"]
    return d


def wfc_summary_table(weibull_df, return_periods=None):
    """Interpolate the flow value at standard return-period thresholds from
    the full Weibull point set."""
    if return_periods is None:
        return_periods = WFC_TABLE_RETURN_PERIODS
    probs = [100.0 / t for t in return_periods]
    x = weibull_df["exceedance_prob_pct"].values  # increases with rank
    y = weibull_df["flow_cfs"].values              # decreases with rank
    flows = np.interp(probs, x, y)
    out = pd.DataFrame({
        "Return Period (years)": return_periods,
        "Percent of Time Equaled or Exceeded (%)": np.round(probs, 2),
        "Flow (cfs)": np.round(flows, 1),
    })
    return out


def wfc_build_display_table(weibull_df, start_wy, end_wy,
                             start_month, start_day, end_month, end_day):
    """Prepend an info block (water years / month-day window / day count
    actually analyzed) on top of the standard return-period summary table."""
    season_text = (
        f"{calendar.month_abbr[start_month]} {start_day} to "
        f"{calendar.month_abbr[end_month]} {end_day}"
    )
    info = pd.DataFrame({
        "Return Period (years)": ["—", "—", "—"],
        "Percent of Time Equaled or Exceeded (%)": ["—", "—", "—"],
        "Flow (cfs)": [
            f"Water years included: WY{start_wy}-WY{end_wy}",
            f"Month/day window (each year): {season_text}",
            f"Days analyzed: {len(weibull_df):,}",
        ],
    })
    summary = wfc_summary_table(weibull_df)
    return pd.concat([info, summary], ignore_index=True)


def wfc_flow_at_percent(weibull_df, pct):
    """Interpolate the combined flow value at a single arbitrary exceedance
    percentage (0-100)."""
    x = weibull_df["exceedance_prob_pct"].values
    y = weibull_df["flow_cfs"].values
    return float(np.interp(pct, x, y))


# -----------------------------------------------------------------------------
# Widgets
# -----------------------------------------------------------------------------
wfc_site1_id = widgets.Text(value="", description="Station 1 ID:",
                             placeholder="e.g. 11169500",
                             style={"description_width": "100px"},
                             layout=widgets.Layout(width="260px"))
wfc_site2_id = widgets.Text(value="", description="Station 2 ID:",
                             placeholder="e.g. 11169800",
                             style={"description_width": "100px"},
                             layout=widgets.Layout(width="260px"))
wfc_label = widgets.Text(value="", description="Label (opt):",
                          placeholder="e.g. Combined flow above confluence",
                          style={"description_width": "90px"},
                          layout=widgets.Layout(width="400px"))

wfc_fetch_button = widgets.Button(description="Fetch & Combine Stations", button_style="info",
                                   layout=widgets.Layout(width="200px"))
wfc_fetch_status = widgets.Label(value="")

# -- Water-year range slider (the "slidy thing"), populated once data is
#    fetched & combined --------------------------------------------------
_wfc_today = dt.date.today()
wfc_year_slider = widgets.IntRangeSlider(
    value=[1990, _wfc_today.year], min=1900, max=_wfc_today.year + 1, step=1,
    description="Water years:", continuous_update=False, disabled=True,
    style={"description_width": "90px"}, layout=widgets.Layout(width="480px"),
)

# -- Recurring month/day window (no year), applied inside every selected
#    water year -----------------------------------------------------------
wfc_start_month = widgets.Dropdown(options=WFC_MONTH_OPTIONS, value=10,
                                    description="Start month:",
                                    style={"description_width": "90px"},
                                    layout=widgets.Layout(width="180px"))
wfc_start_day = widgets.Dropdown(options=WFC_DAY_OPTIONS, value=1,
                                  description="day:",
                                  style={"description_width": "35px"},
                                  layout=widgets.Layout(width="90px"))
wfc_end_month = widgets.Dropdown(options=WFC_MONTH_OPTIONS, value=9,
                                  description="End month:",
                                  style={"description_width": "90px"},
                                  layout=widgets.Layout(width="180px"))
wfc_end_day = widgets.Dropdown(options=WFC_DAY_OPTIONS, value=30,
                                description="day:",
                                style={"description_width": "35px"},
                                layout=widgets.Layout(width="90px"))

wfc_log_toggle = widgets.Checkbox(value=True, description="Log scale (y-axis)",
                                   style={"description_width": "initial"})
wfc_curve_color = widgets.ColorPicker(value=WRA_NAVY, description="Curve color:",
                                       style={"description_width": "85px"},
                                       layout=widgets.Layout(width="220px"))
wfc_curve_style = widgets.Dropdown(options=WFC_LINESTYLE_OPTIONS, value="-",
                                    description="Line style:",
                                    style={"description_width": "70px"},
                                    layout=widgets.Layout(width="160px"))
wfc_show_points_toggle = widgets.Checkbox(value=False, description="Show individual daily points",
                                           style={"description_width": "initial"})
wfc_show_markers_toggle = widgets.Checkbox(value=True, description="Mark standard return periods",
                                            style={"description_width": "initial"})

# --- User-adjustable percentage threshold line (default 50%) --------------
wfc_threshold_toggle = widgets.Checkbox(
    value=True, description="Show flow value at a chosen percentage",
    style={"description_width": "initial"},
)
wfc_threshold_pct = widgets.BoundedFloatText(
    value=50.0, min=0.01, max=99.99, step=0.5,
    description="Percentage:", style={"description_width": "80px"},
    layout=widgets.Layout(width="180px"),
)
wfc_threshold_color = widgets.ColorPicker(value="#B22222", description="Line color:",
                                           style={"description_width": "80px"},
                                           layout=widgets.Layout(width="210px"))
wfc_threshold_style = widgets.Dropdown(options=WFC_LINESTYLE_OPTIONS, value="--",
                                        description="Line style:",
                                        style={"description_width": "70px"},
                                        layout=widgets.Layout(width="160px"))
wfc_threshold_box = widgets.HBox([wfc_threshold_pct, wfc_threshold_color, wfc_threshold_style])


def _wfc_toggle_threshold_box(change):
    wfc_threshold_box.layout.display = "" if change["new"] else "none"


wfc_threshold_toggle.observe(_wfc_toggle_threshold_box, names="value")

wfc_title_input = widgets.Text(
    value="", description="Title (opt):",
    placeholder="e.g. Combined Flow-Duration Curve",
    style={"description_width": "90px"}, layout=widgets.Layout(width="500px"),
)

wfc_run_button = widgets.Button(description="Run Analysis", button_style="primary",
                                 layout=widgets.Layout(width="130px"))
wfc_save_svg_button = widgets.Button(description="Save Plot SVG", button_style="success",
                                      layout=widgets.Layout(width="130px"))
wfc_save_csv_button = widgets.Button(description="Save Full Table CSV", button_style="success",
                                      layout=widgets.Layout(width="160px"))
wfc_status = widgets.Label(value="")
wfc_out = widgets.Output()

_wfc_header_html = (
    "<div style='display:flex;align-items:center;gap:14px;"
    "background:white;padding:10px 20px;border-radius:6px;"
    f"border:2px solid {WRA_NAVY};margin-bottom:6px'>"
    + (f"<img src='data:image/png;base64,{_LOGO_B64}' "
       "style='height:42px;width:auto;display:block' alt='WRA logo'>"
       if _LOGO_B64 else "")
    + f"<span style='color:{WRA_NAVY};font-size:16px;"
      "font-family:Verdana;font-weight:bold'>"
      "Weibull Flow-Duration Analysis — Combined Two-Station Flow "
      "vs. Exceedance Probability</span></div>"
)

wfc_ui = widgets.VBox([
    widgets.HTML(_wfc_header_html),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([wfc_site1_id, wfc_site2_id]),
    widgets.HBox([wfc_label]),
    widgets.HBox([wfc_fetch_button, wfc_fetch_status]),
    widgets.HTML("<b>Water years to include:</b>"),
    wfc_year_slider,
    widgets.HTML("<b>Month/day window to apply within each selected water year:</b>"),
    widgets.HBox([wfc_start_month, wfc_start_day]),
    widgets.HBox([wfc_end_month, wfc_end_day]),
    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([wfc_log_toggle, wfc_show_points_toggle, wfc_show_markers_toggle]),
    widgets.HBox([wfc_curve_color, wfc_curve_style]),
    wfc_threshold_toggle,
    wfc_threshold_box,
    widgets.HBox([wfc_title_input]),
    widgets.HBox([wfc_run_button, wfc_save_svg_button, wfc_save_csv_button, wfc_status]),
    wfc_out,
])

# -----------------------------------------------------------------------------
# State held between callbacks (fetched/combined data, computed tables, figure)
# -----------------------------------------------------------------------------
_wfc_state = {
    "combined_df": None,
    "site1": None,
    "site2": None,
    "combined_label": None,
    "weibull_full": None,
    "current_fig": None,
}


# -----------------------------------------------------------------------------
# Fetch & combine callback
# -----------------------------------------------------------------------------
def wfc_on_fetch(b):
    site1 = wfc_site1_id.value.strip()
    site2 = wfc_site2_id.value.strip()
    if not site1 or not site2:
        wfc_fetch_status.value = "Enter both Station 1 and Station 2 IDs."
        return
    try:
        wfc_fetch_status.value = f"Fetching full daily history for Station {site1}..."
        full1 = wfc_fetch_full_daily_flow(site1)

        wfc_fetch_status.value = f"Fetching full daily history for Station {site2}..."
        full2 = wfc_fetch_full_daily_flow(site2)

        wfc_fetch_status.value = "Combining both stations' daily flow..."
        combined = wfc_combine_two_stations(full1, full2)

        name1 = wfc_fetch_station_name(site1)
        name2 = wfc_fetch_station_name(site2)
        combined_label = wfc_label.value.strip() or f"{name1} + {name2}"

        wy_all = combined["date"].apply(wfc_water_year_of)
        wy_min, wy_max = int(wy_all.min()), int(wy_all.max())

        wfc_year_slider.min = wy_min
        wfc_year_slider.max = wy_max
        wfc_year_slider.value = [wy_min, wy_max]
        wfc_year_slider.disabled = False

        _wfc_state["combined_df"] = combined
        _wfc_state["site1"] = site1
        _wfc_state["site2"] = site2
        _wfc_state["combined_label"] = combined_label

        wfc_fetch_status.value = (
            f"Loaded {combined_label} — overlapping record WY{wy_min}-WY{wy_max}, "
            f"{len(combined):,} combined daily values."
        )
    except Exception as e:
        wfc_fetch_status.value = f"Error: {e}"
        raise


wfc_fetch_button.on_click(wfc_on_fetch)


# -----------------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------------
def wfc_make_plot(weibull_df, combined_label, start_wy, end_wy,
                   start_month, start_day, end_month, end_day,
                   use_log, curve_color, curve_style, show_points, show_markers,
                   show_threshold, threshold_pct, threshold_color, threshold_style,
                   custom_title=""):
    fig, ax = plt.subplots(figsize=(11, 6.5))

    if show_points:
        ax.scatter(weibull_df["exceedance_prob_pct"], weibull_df["flow_cfs"],
                   s=6, color="#AAAAAA", alpha=0.5, zorder=1,
                   label="Individual daily values")

    ax.plot(weibull_df["exceedance_prob_pct"], weibull_df["flow_cfs"],
            color=curve_color, linewidth=1.8, linestyle=curve_style,
            zorder=2, label="Combined flow-duration curve (Weibull)")

    if show_markers:
        summary = wfc_summary_table(weibull_df, return_periods=WFC_PLOT_MARKER_RETURN_PERIODS)
        ax.scatter(summary["Percent of Time Equaled or Exceeded (%)"],
                   summary["Flow (cfs)"], color=WRA_GOLD, edgecolor=WRA_NAVY,
                   s=40, zorder=3, label="Standard return periods (2-100 yr)")

    if use_log:
        ax.set_yscale("log")

    ax.set_xlim(0, 100)
    ax.set_xlabel("Percent of Time Combined Flow Was Equaled or Exceeded (%)")
    ax.set_ylabel("Combined Flow (cfs)")

    # -- User-chosen percentage threshold: dashed drop-lines + callout box --
    if show_threshold:
        flow_at_pct = wfc_flow_at_percent(weibull_df, threshold_pct)
        ylim = ax.get_ylim()

        ax.plot([threshold_pct, threshold_pct], [ylim[0], flow_at_pct],
                color=threshold_color, linestyle=threshold_style,
                linewidth=1.6, zorder=4)
        ax.plot([0, threshold_pct], [flow_at_pct, flow_at_pct],
                color=threshold_color, linestyle=threshold_style,
                linewidth=1.2, alpha=0.75, zorder=4)
        ax.set_ylim(ylim)

        label_text = f"Q{threshold_pct:g}% = {flow_at_pct:,.1f} cfs"
        if threshold_pct > 82:
            xytext = (threshold_pct - 3, flow_at_pct)
            ha = "right"
        else:
            xytext = (threshold_pct + 3, flow_at_pct)
            ha = "left"
        ax.annotate(
            label_text, xy=(threshold_pct, flow_at_pct), xytext=xytext,
            va="center", ha=ha, fontsize=9, fontweight="bold", color=WRA_NAVY,
            bbox=dict(boxstyle="round,pad=0.35", fc="white",
                      ec=threshold_color, lw=1.3), zorder=5,
        )

    season_text = (
        f"{calendar.month_abbr[start_month]} {start_day} to "
        f"{calendar.month_abbr[end_month]} {end_day}"
    )
    default_title = (
        f"Combined Flow-Duration Curve (Weibull Plotting Position)\n"
        f"{combined_label} — WY{start_wy}-WY{end_wy} — {season_text} each year"
    )
    ax.set_title(custom_title.strip() if custom_title.strip() else default_title, fontsize=11)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)

    fig.text(0.99, 0.01, f"© {dt.date.today().year} WRA, Inc.",
              ha="right", va="bottom", fontsize=7,
              fontfamily="Verdana", color="#888888")

    fig.tight_layout()
    _wfc_state["current_fig"] = fig
    plt.show()


# -----------------------------------------------------------------------------
# Run callback
# -----------------------------------------------------------------------------
def wfc_on_run(b):
    with wfc_out:
        clear_output(wait=True)
        try:
            combined_df = _wfc_state["combined_df"]
            if combined_df is None:
                wfc_status.value = "Fetch and combine both stations first."
                return

            start_wy, end_wy = wfc_year_slider.value
            start_month, start_day = wfc_start_month.value, wfc_start_day.value
            end_month, end_day = wfc_end_month.value, wfc_end_day.value

            wfc_status.value = "Filtering combined data..."
            filtered = wfc_filter_by_years_and_season(
                combined_df, start_wy, end_wy, start_month, start_day, end_month, end_day
            )
            if filtered.empty:
                raise ValueError(
                    "No combined daily values fall within the selected water "
                    "years and month/day window."
                )

            wfc_status.value = "Computing Weibull plotting positions..."
            weibull_full = wfc_compute_weibull_table(filtered)
            _wfc_state["weibull_full"] = weibull_full

            combined_label = _wfc_state["combined_label"]

            wfc_status.value = "Rendering chart..."
            wfc_make_plot(
                weibull_full, combined_label, start_wy, end_wy,
                start_month, start_day, end_month, end_day,
                wfc_log_toggle.value, wfc_curve_color.value, wfc_curve_style.value,
                wfc_show_points_toggle.value, wfc_show_markers_toggle.value,
                wfc_threshold_toggle.value, wfc_threshold_pct.value,
                wfc_threshold_color.value, wfc_threshold_style.value,
                wfc_title_input.value,
            )

            print(f"\n{combined_label}")
            print("Flow-Duration Summary Table (Weibull plotting position, "
                  "interpolated at standard return periods):\n")
            display(wfc_build_display_table(
                weibull_full, start_wy, end_wy, start_month, start_day, end_month, end_day
            ))

            if wfc_threshold_toggle.value:
                flow_at_pct = wfc_flow_at_percent(weibull_full, wfc_threshold_pct.value)
                print(f"\nFlow equaled or exceeded {wfc_threshold_pct.value:g}% of the time: "
                      f"{flow_at_pct:,.1f} cfs")

            wfc_status.value = (
                f"Done — n={len(weibull_full):,} combined daily values retained. "
                "Use 'Save Full Table CSV' to export every point."
            )
        except Exception as e:
            wfc_status.value = f"Error: {e}"
            raise


wfc_run_button.on_click(wfc_on_run)


# -----------------------------------------------------------------------------
# Save callbacks
# -----------------------------------------------------------------------------
def wfc_on_save_svg(b):
    fig = _wfc_state["current_fig"]
    if fig is None:
        wfc_status.value = "Run the analysis first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save combined flow-duration curve as SVG",
            defaultextension=".svg",
            filetypes=[("SVG files", "*.svg"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            fig.savefig(path, format="svg", bbox_inches="tight")
            wfc_status.value = f"Saved plot: {path}"
        else:
            wfc_status.value = "Save cancelled."
    except Exception as e:
        wfc_status.value = f"Save error: {e}"


def wfc_on_save_csv(b):
    weibull_full = _wfc_state["weibull_full"]
    if weibull_full is None:
        wfc_status.value = "Run the analysis first."
        return
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.asksaveasfilename(
            title="Save full combined Weibull flow-duration table as CSV",
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
        )
        root.destroy()
        if path:
            out_cols = ["date", "flow_cfs", "water_year", "rank",
                        "exceedance_prob_pct", "return_period_years"]
            weibull_full[out_cols].to_csv(path, index=False)
            wfc_status.value = f"Saved full table ({len(weibull_full):,} rows): {path}"
        else:
            wfc_status.value = "Save cancelled."
    except Exception as e:
        wfc_status.value = f"Save error: {e}"


wfc_save_svg_button.on_click(wfc_on_save_svg)
wfc_save_csv_button.on_click(wfc_on_save_csv)

# -----------------------------------------------------------------------------
# Launch
# -----------------------------------------------------------------------------
display(wfc_ui)